# BLOK 4 — Wdrożenie i użycie modelu

**Struktura notebooka**

- Opis otwierający — Operacyjne wdrożenie modelu klasyfikacyjnego oraz budowa warstwy scoringowej dla systemu relokacji floty rowerowej

- Dział 4.1 — Zapis danych do bazy danych
  - 4.1.1 — Definicja środowiska BLOKU 4 i kontraktu ścieżek
  - 4.1.2 — Utworzenie lokalnej bazy scoringowej
  - 4.1.3 — Utworzenie tabel warstwy scoringowej
  - 4.1.4 — Zapis artefaktów sekcji 4.1

- Dział 4.2 — Pobieranie modelu i konfiguracji inferencyjnej
  - 4.2.1 — Definicja i walidacja pakietu modelowego
  - 4.2.2 — Wczytanie kontraktu inferencyjnego i metadanych modelu
  - 4.2.3 — Wczytanie obiektu modelowego
  - 4.2.4 — Zapis artefaktów sekcji 4.2

- Dział 4.3 — Przygotowanie wsadu danych do predykcji na kolejny dzień
  - 4.3.1 — Definicja źródeł wsadu inferencyjnego i trybu wyboru daty scoringowej
  - 4.3.2 — Ustalenie finalnej daty scoringu
  - 4.3.3 — Wczytanie batcha scoringowego dla wybranej daty
  - 4.3.4 — Walidacja kolejności cech i budowa macierzy wejściowej modelu
  - 4.3.5 — Walidacja typów danych i nullowości cech modelowych
  - 4.3.6 — Walidacja anti-leakage i finalnej gotowości wsadu scoringowego
  - 4.3.7 — Zapis finalnego wsadu scoringowego i kontraktu sekcji 4.3

- Dział 4.4 — Predykcja dzienna
  - 4.4.1 — Wczytanie wsadu scoringowego i kontraktu predykcji
  - 4.4.2 — Uruchomienie scoringu dziennego i budowa tabeli predykcji
  - 4.4.3 — Zapis wyników predykcji i kontraktu sekcji 4.4

- Dział 4.5 — Zapis wyników i ich prezentacja w aplikacji
  - 4.5.1 — Przygotowanie zbioru wynikowego dla aplikacji
  - 4.5.2 — Zapis zbioru wynikowego dla aplikacji i handoff sekcji 4.5

## Executive Summary (Główne założenia biznesowe i inżynieryjne)

Niniejszy notatnik stanowi warstwę wdrożeniową systemu relokacji floty rowerowej. Celem BLOKU 4 jest operacyjne użycie wcześniej wytrenowanego modelu klasyfikacyjnego oraz budowa kompletnego pipeline'u inferencyjnego dla predykcji dziennej. BLOK 4 odpowiada za przygotowanie środowiska scoringowego, walidację wsadu inferencyjnego, uruchomienie predykcji oraz zapis wyników do warstwy aplikacyjnej i operacyjnej.

**Kluczowe założenia analityczne i architektoniczne:**

1. **Pipeline inferencyjny:** BLOK 4 wykorzystuje gotowy model oraz finalny kontrakt cech wypracowany w BLOKU 3. Etap nie obejmuje ponownego treningu modeli.

2. **Ścisły kontrakt modelowy:** Inferencja wykonywana jest wyłącznie na cechach zgodnych z finalnym `keep_cols`, zachowując pełną zgodność kolejności, typów danych oraz polityki null semantics.

3. **Anti-Leakage:** Predykcja dzienna wykonywana jest wyłącznie na wsadzie dostępnym przed początkiem dnia scoringowego. Pipeline nie korzysta z danych powstałych po momencie decyzji operacyjnej.

4. **Warstwa scoringowa:** BLOK 4 buduje lokalną warstwę scoringową opartą o zapis batchy inferencyjnych, wyników predykcji oraz artefaktów deploymentowych.

5. **Predykcja operacyjna:** Wyniki scoringu służą do budowy listy stacji wysokiego ryzyka i stanowią podstawę dla aplikacji dyspozytorskiej oraz późniejszych rekomendacji relokacyjnych.

6. **Deployment i handoff:** Finalnym rezultatem BLOKU 4 są gotowe artefakty wdrożeniowe, batch scoringowy, tabela predykcji oraz warstwa danych przekazywana do aplikacji użytkowej.

---

## Struktura Notatnika

**4.1 Zapis danych do bazy danych**  
Budowa środowiska deploymentowego oraz lokalnej warstwy scoringowej. Etap obejmuje definicję kontraktu ścieżek, przygotowanie lokalnej bazy scoringowej, utworzenie tabel inferencyjnych oraz zapis artefaktów wdrożeniowych sekcji 4.1.

**4.2 Pobieranie modelu i konfiguracji inferencyjnej**  
Wczytanie finalnego pakietu modelowego i kontraktu inferencyjnego. Etap obejmuje walidację artefaktów modelowych, konfiguracji scoringu, metadanych deploymentowych oraz inicjalizację środowiska inferencyjnego.

**4.3 Przygotowanie wsadu danych do predykcji na kolejny dzień**  
Budowa batcha scoringowego dla dnia T. Etap obejmuje wybór daty scoringowej, wczytanie danych inferencyjnych, kontrolę typów danych, walidację kolejności cech, kontrolę null semantics oraz finalną walidację anti-leakage.

**4.4 Predykcja dzienna**  
Uruchomienie inferencji modelowej dla wsadu dziennego. Etap obejmuje scoring probabilistyczny, budowę tabeli predykcji oraz zapis wyników inferencyjnych wraz z kontraktem sekcji 4.4.

**4.5 Zapis wyników i ich prezentacja w aplikacji**  
Przygotowanie finalnej warstwy aplikacyjnej. Etap obejmuje budowę datasetu wynikowego dla aplikacji operacyjnej, zapis wyników scoringu oraz przygotowanie warstwy handoff dla panelu dyspozytorskiego.

## BLOK 4 - Wdrożenie i użycie modelu

## 4.1 Zapis danych do bazy danych

In [1]:
# 4.1.1 Definicja środowiska BLOKU 4 i kontraktu ścieżek

from pathlib import Path
import pandas as pd

# Search for the Block 4 working directory
project_root_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path("/root/projects/6_samodzielny_projekt_koncowy_wsl/4_Wdrozenie_i_uzycie_modelu").resolve(),
]

step4u_1_project_root = None

for candidate in project_root_candidates:
    if (
        candidate.exists()
        and (candidate / "4_Wdrozenie_i_uzycie_modelu.ipynb").exists()
        and (candidate / "aplikacja_dzien_stacja.py").exists()
        and (candidate / "input_model_package").exists()
        and (candidate / "db_scoring").exists()
        and (candidate / "outputs_dzien_stacja").exists()
    ):
        step4u_1_project_root = candidate
        break

assert step4u_1_project_root is not None, "Block 4 project root was not found."

# Define Block 4 paths
step4u_1_notebook_path = step4u_1_project_root / "4_Wdrozenie_i_uzycie_modelu.ipynb"
step4u_1_app_path = step4u_1_project_root / "aplikacja_dzien_stacja.py"
step4u_1_input_model_package_dir = step4u_1_project_root / "input_model_package"
step4u_1_db_dir = step4u_1_project_root / "db_scoring"
step4u_1_outputs_dir = step4u_1_project_root / "outputs_dzien_stacja"

# Define scoring database layer
step4u_1_db_engine_name = "sqlite"
step4u_1_db_path = step4u_1_db_dir / "b4u_scoring_store.sqlite"

# Define Section 4.1 artifacts
step4u_1_db_inventory_artifact_path = step4u_1_outputs_dir / "b4u_01_db_inventory.parquet"
step4u_1_db_contract_artifact_path = step4u_1_outputs_dir / "b4u_01_db_contract.json"

# Define required input package artifacts
step4u_1_required_input_artifacts = [
    "b4_15_best_model.joblib",
    "b4_15_inference_config.json",
    "b4_15_model_registry.json",
    "b4_15_handoff_manifest.json",
    "b4_15_feature_importance.parquet",
]

# Validate required paths
required_paths = [
    step4u_1_notebook_path,
    step4u_1_app_path,
    step4u_1_input_model_package_dir,
    step4u_1_db_dir,
    step4u_1_outputs_dir,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f"Missing required paths: {missing_paths}"

# Build input package inventory
step4u_1_input_inventory_rows = []

for artifact_name in step4u_1_required_input_artifacts:
    artifact_path = step4u_1_input_model_package_dir / artifact_name
    step4u_1_input_inventory_rows.append(
        {
            "artifact_name": artifact_name,
            "artifact_path": str(artifact_path),
            "exists": int(artifact_path.exists()),
        }
    )

step4u_1_input_inventory_df = pd.DataFrame(step4u_1_input_inventory_rows)

missing_input_artifacts = step4u_1_input_inventory_df.loc[
    step4u_1_input_inventory_df["exists"] == 0, "artifact_name"
].tolist()
assert not missing_input_artifacts, f"Missing input model artifacts: {missing_input_artifacts}"

# Define scoring layer table contract
step4u_1_db_table_contract_df = pd.DataFrame(
    [
        {
            "table_name": "scoring_input_next_day",
            "table_role": "next_day_scoring_input",
            "grain": "day_station",
            "write_stage": "4.3",
            "primary_keys": "activity_date|station_id",
        },
        {
            "table_name": "predictions_next_day",
            "table_role": "next_day_prediction_output",
            "grain": "day_station",
            "write_stage": "4.4",
            "primary_keys": "activity_date|station_id|scoring_timestamp",
        },
    ]
)

# Build Section 4.1 environment inventory
step4u_1_db_inventory_df = pd.DataFrame(
    [
        {
            "project_root": str(step4u_1_project_root),
            "notebook_path": str(step4u_1_notebook_path),
            "app_path": str(step4u_1_app_path),
            "input_model_package_dir": str(step4u_1_input_model_package_dir),
            "db_dir": str(step4u_1_db_dir),
            "outputs_dir": str(step4u_1_outputs_dir),
            "db_engine_name": step4u_1_db_engine_name,
            "db_path": str(step4u_1_db_path),
            "db_file_exists": int(step4u_1_db_path.exists()),
            "input_artifact_count": int(len(step4u_1_required_input_artifacts)),
            "table_contract_count": int(step4u_1_db_table_contract_df.shape[0]),
        }
    ]
)

display(step4u_1_db_inventory_df)
display(step4u_1_input_inventory_df)
display(step4u_1_db_table_contract_df)

,project_root,notebook_path,app_path,input_model_package_dir,db_dir,outputs_dir,db_engine_name,db_path,db_file_exists,input_artifact_count,table_contract_count
0,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,/root/projects/6_samodzielny_projekt_koncowy_w...,sqlite,/root/projects/6_samodzielny_projekt_koncowy_w...,1,5,2


,artifact_name,artifact_path,exists
0,b4_15_best_model.joblib,/root/projects/6_samodzielny_projekt_koncowy_w...,1
1,b4_15_inference_config.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1
2,b4_15_model_registry.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1
3,b4_15_handoff_manifest.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1
4,b4_15_feature_importance.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1


,table_name,table_role,grain,write_stage,primary_keys
0,scoring_input_next_day,next_day_scoring_input,day_station,4.3,activity_date|station_id
1,predictions_next_day,next_day_prediction_output,day_station,4.4,activity_date|station_id|scoring_timestamp


In [2]:
# 4.1.2 Utworzenie lokalnej bazy scoringowej

import sqlite3
from pathlib import Path
import pandas as pd

# Validate required objects from 4.1.1
required_objects = [
    "step4u_1_db_engine_name",
    "step4u_1_db_path",
    "step4u_1_db_dir",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from 4.1.1: {missing_objects}"

assert step4u_1_db_engine_name == "sqlite", (
    f"Unsupported database engine: {step4u_1_db_engine_name}"
)

# Create local scoring database file
step4u_2_db_path = Path(step4u_1_db_path)
step4u_2_db_path.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(step4u_2_db_path) as connection:
    step4u_2_sqlite_version = connection.execute(
        "SELECT sqlite_version()"
    ).fetchone()[0]

    step4u_2_existing_tables = connection.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """
    ).fetchall()

step4u_2_existing_table_names = [row[0] for row in step4u_2_existing_tables]
step4u_2_database_ready = int(step4u_2_db_path.exists())

step4u_2_db_readiness_df = pd.DataFrame(
    [
        {
            "db_engine_name": step4u_1_db_engine_name,
            "db_path": str(step4u_2_db_path),
            "db_file_exists": int(step4u_2_db_path.exists()),
            "sqlite_version": step4u_2_sqlite_version,
            "existing_table_count": int(len(step4u_2_existing_table_names)),
            "database_ready": step4u_2_database_ready,
        }
    ]
)

display(step4u_2_db_readiness_df)

,db_engine_name,db_path,db_file_exists,sqlite_version,existing_table_count,database_ready
0,sqlite,/root/projects/6_samodzielny_projekt_koncowy_w...,1,3.45.1,2,1


In [3]:
# 4.1.3 Utworzenie tabel warstwy scoringowej

import sqlite3
import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_1_db_path",
    "step4u_1_db_table_contract_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

step4u_3_db_path = step4u_1_db_path

# Define scoring layer table names
step4u_3_scoring_input_table_name = "scoring_input_next_day"
step4u_3_predictions_table_name = "predictions_next_day"

# Create scoring layer tables
step4u_3_schema_sql = f"""
CREATE TABLE IF NOT EXISTS {step4u_3_scoring_input_table_name} (
    activity_date TEXT NOT NULL,
    station_id TEXT NOT NULL,
    scoring_input_created_at TEXT NOT NULL,
    feature_payload_json TEXT NOT NULL,
    PRIMARY KEY (activity_date, station_id)
);

CREATE TABLE IF NOT EXISTS {step4u_3_predictions_table_name} (
    activity_date TEXT NOT NULL,
    station_id TEXT NOT NULL,
    scoring_timestamp TEXT NOT NULL,
    model_version TEXT NOT NULL,
    applied_threshold REAL NOT NULL,
    predicted_probability REAL NOT NULL,
    predicted_label INTEGER NOT NULL,
    prediction_created_at TEXT NOT NULL,
    PRIMARY KEY (activity_date, station_id, scoring_timestamp)
);
"""

with sqlite3.connect(step4u_3_db_path) as connection:
    connection.executescript(step4u_3_schema_sql)

    step4u_3_existing_tables_raw = connection.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """
    ).fetchall()

    step4u_3_table_schema_rows = []

    for table_name_row in step4u_3_existing_tables_raw:
        table_name = table_name_row[0]

        if table_name not in {
            step4u_3_scoring_input_table_name,
            step4u_3_predictions_table_name,
        }:
            continue

        table_info_rows = connection.execute(
            f"PRAGMA table_info({table_name})"
        ).fetchall()

        for column_row in table_info_rows:
            step4u_3_table_schema_rows.append(
                {
                    "table_name": table_name,
                    "column_position": int(column_row[0]),
                    "column_name": column_row[1],
                    "column_type": column_row[2],
                    "not_null": int(column_row[3]),
                    "default_value": column_row[4],
                    "primary_key_position": int(column_row[5]),
                }
            )

step4u_3_existing_table_names = [row[0] for row in step4u_3_existing_tables_raw]

step4u_3_db_table_inventory_df = pd.DataFrame(
    [
        {
            "table_name": row["table_name"],
            "table_exists": int(row["table_name"] in step4u_3_existing_table_names),
            "grain": row["grain"],
            "write_stage": row["write_stage"],
            "primary_keys": row["primary_keys"],
        }
        for _, row in step4u_1_db_table_contract_df.iterrows()
    ]
)

step4u_3_db_schema_df = pd.DataFrame(step4u_3_table_schema_rows).sort_values(
    ["table_name", "column_position"]
).reset_index(drop=True)

display(step4u_3_db_table_inventory_df)
display(step4u_3_db_schema_df)

,table_name,table_exists,grain,write_stage,primary_keys
0,scoring_input_next_day,1,day_station,4.3,activity_date|station_id
1,predictions_next_day,1,day_station,4.4,activity_date|station_id|scoring_timestamp


,table_name,column_position,column_name,column_type,not_null,default_value,primary_key_position
0,predictions_next_day,0,activity_date,TEXT,1,None,1
1,predictions_next_day,1,station_id,TEXT,1,None,2
2,predictions_next_day,2,scoring_timestamp,TEXT,1,None,3
3,predictions_next_day,3,model_version,TEXT,1,None,0
4,predictions_next_day,4,applied_threshold,REAL,1,None,0
5,predictions_next_day,5,predicted_probability,REAL,1,None,0
6,predictions_next_day,6,predicted_label,INTEGER,1,None,0
7,predictions_next_day,7,prediction_created_at,TEXT,1,None,0
8,scoring_input_next_day,0,activity_date,TEXT,1,None,1
9,scoring_input_next_day,1,station_id,TEXT,1,None,2


In [4]:
# 4.1.4 Zapis artefaktów sekcji 4.1

import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_1_db_inventory_artifact_path",
    "step4u_1_db_contract_artifact_path",
    "step4u_1_db_engine_name",
    "step4u_1_db_path",
    "step4u_1_input_inventory_df",
    "step4u_1_db_table_contract_df",
    "step4u_2_db_readiness_df",
    "step4u_3_db_table_inventory_df",
    "step4u_3_db_schema_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

step4u_4_db_inventory_artifact_path = Path(step4u_1_db_inventory_artifact_path)
step4u_4_db_contract_artifact_path = Path(step4u_1_db_contract_artifact_path)

step4u_4_db_inventory_artifact_path.parent.mkdir(parents=True, exist_ok=True)
step4u_4_db_contract_artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Build unified inventory artifact
step4u_4_inventory_rows = []

step4u_4_inventory_rows.append(
    {
        "inventory_section": "database_runtime",
        "item_name": "scoring_database",
        "item_type": "database",
        "item_path": str(step4u_1_db_path),
        "exists": int(step4u_2_db_readiness_df.loc[0, "db_file_exists"]),
        "detail_json": json.dumps(
            {
                "db_engine_name": step4u_1_db_engine_name,
                "sqlite_version": str(step4u_2_db_readiness_df.loc[0, "sqlite_version"]),
                "existing_table_count": int(step4u_2_db_readiness_df.loc[0, "existing_table_count"]),
                "database_ready": int(step4u_2_db_readiness_df.loc[0, "database_ready"]),
            },
            ensure_ascii=False,
        ),
    }
)

for _, row in step4u_1_input_inventory_df.iterrows():
    step4u_4_inventory_rows.append(
        {
            "inventory_section": "input_model_package",
            "item_name": row["artifact_name"],
            "item_type": "input_artifact",
            "item_path": row["artifact_path"],
            "exists": int(row["exists"]),
            "detail_json": json.dumps({}, ensure_ascii=False),
        }
    )

for _, row in step4u_3_db_table_inventory_df.iterrows():
    step4u_4_inventory_rows.append(
        {
            "inventory_section": "database_tables",
            "item_name": row["table_name"],
            "item_type": "table",
            "item_path": str(step4u_1_db_path),
            "exists": int(row["table_exists"]),
            "detail_json": json.dumps(
                {
                    "grain": row["grain"],
                    "write_stage": row["write_stage"],
                    "primary_keys": row["primary_keys"],
                },
                ensure_ascii=False,
            ),
        }
    )

for _, row in step4u_3_db_schema_df.iterrows():
    step4u_4_inventory_rows.append(
        {
            "inventory_section": "database_schema",
            "item_name": f"{row['table_name']}.{row['column_name']}",
            "item_type": "column",
            "item_path": str(step4u_1_db_path),
            "exists": 1,
            "detail_json": json.dumps(
                {
                    "table_name": row["table_name"],
                    "column_position": int(row["column_position"]),
                    "column_name": row["column_name"],
                    "column_type": row["column_type"],
                    "not_null": int(row["not_null"]),
                    "default_value": row["default_value"],
                    "primary_key_position": int(row["primary_key_position"]),
                },
                ensure_ascii=False,
            ),
        }
    )

step4u_4_db_inventory_df = pd.DataFrame(step4u_4_inventory_rows)

step4u_4_db_inventory_df["inventory_section"] = step4u_4_db_inventory_df["inventory_section"].astype("string")
step4u_4_db_inventory_df["item_name"] = step4u_4_db_inventory_df["item_name"].astype("string")
step4u_4_db_inventory_df["item_type"] = step4u_4_db_inventory_df["item_type"].astype("string")
step4u_4_db_inventory_df["item_path"] = step4u_4_db_inventory_df["item_path"].astype("string")
step4u_4_db_inventory_df["exists"] = step4u_4_db_inventory_df["exists"].astype("int8")
step4u_4_db_inventory_df["detail_json"] = step4u_4_db_inventory_df["detail_json"].astype("string")

step4u_4_inventory_table = pa.Table.from_pandas(
    step4u_4_db_inventory_df,
    preserve_index=False,
)
pq.write_table(
    step4u_4_inventory_table,
    step4u_4_db_inventory_artifact_path,
)

# Build database contract artifact
step4u_4_db_contract_payload = {
    "section_name": "4.1",
    "db_engine_name": step4u_1_db_engine_name,
    "db_path": str(step4u_1_db_path),
    "db_file_exists": int(step4u_2_db_readiness_df.loc[0, "db_file_exists"]),
    "database_ready": int(step4u_2_db_readiness_df.loc[0, "database_ready"]),
    "sqlite_version": str(step4u_2_db_readiness_df.loc[0, "sqlite_version"]),
    "input_model_package_artifacts": step4u_1_input_inventory_df.to_dict(orient="records"),
    "table_contract": step4u_1_db_table_contract_df.to_dict(orient="records"),
    "table_inventory": step4u_3_db_table_inventory_df.to_dict(orient="records"),
    "table_schema": step4u_3_db_schema_df.to_dict(orient="records"),
}

with open(step4u_4_db_contract_artifact_path, "w", encoding="utf-8") as file:
    json.dump(step4u_4_db_contract_payload, file, ensure_ascii=False, indent=2)

step4u_4_artifact_save_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4u_01_db_inventory.parquet",
            "artifact_path": str(step4u_4_db_inventory_artifact_path),
            "exists": int(step4u_4_db_inventory_artifact_path.exists()),
            "row_count": int(step4u_4_db_inventory_df.shape[0]),
        },
        {
            "artifact_name": "b4u_01_db_contract.json",
            "artifact_path": str(step4u_4_db_contract_artifact_path),
            "exists": int(step4u_4_db_contract_artifact_path.exists()),
            "row_count": pd.NA,
        },
    ]
)

display(step4u_4_artifact_save_df)

,artifact_name,artifact_path,exists,row_count
0,b4u_01_db_inventory.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1,20
1,b4u_01_db_contract.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1,<NA>


## Podsumowanie działu 4.1 Zapis danych do bazy danych

**Kontekst inżynieryjny:** W dziale `4.1` przygotowano lokalną warstwę bazy danych dla scoringu dziennego w ziarnie `dzień–stacja`. Najpierw zdefiniowano środowisko robocze BLOKU 4, ścieżki do katalogów `input_model_package`, `db_scoring` i `outputs_dzien_stacja` oraz docelową lokalizację bazy `b4u_scoring_store.sqlite`. Następnie zinwentaryzowano wymagany pakiet wejściowy modelu, obejmujący `b4_15_best_model.joblib`, `b4_15_inference_config.json`, `b4_15_model_registry.json`, `b4_15_handoff_manifest.json` oraz `b4_15_feature_importance.parquet`. W kolejnym kroku utworzono lokalną bazę scoringową w SQLite oraz zmaterializowano dwie tabele warstwy operacyjnej: `scoring_input_next_day` i `predictions_next_day`. Na końcu zapisano artefakty sekcji `4.1`: `b4u_01_db_inventory.parquet` oraz `b4u_01_db_contract.json`.

**Interpretacja wyniku:** Warstwa bazy dla BLOKU 4 została przygotowana poprawnie. Lokalny plik bazy scoringowej istnieje i działa w silniku `SQLite 3.45.1`. Pakiet wejściowy modelu jest kompletny i zawiera wszystkie `5` wymaganych artefaktów potrzebnych do dalszego pipeline’u inferencyjnego. W bazie utworzono `2` tabele zgodne z kontraktem procesu: `scoring_input_next_day` dla wsadu do punktu `4.3` oraz `predictions_next_day` dla wyników predykcji z punktu `4.4`. Tabela wejściowa została zdefiniowana z kluczem głównym `activity_date + station_id`, natomiast tabela wynikowa z kluczem `activity_date + station_id + scoring_timestamp`, co zapewnia rozróżnialność kolejnych uruchomień scoringu. Zapisany artefakt `b4u_01_db_inventory.parquet` zawiera `20` rekordów inwentarza obejmujących bazę, artefakty wejściowe, tabele oraz ich schemat.

**Znaczenie biznesowe:** Dział `4.1` ustanawia formalną warstwę przechowywania dla dalszych etapów BLOKU 4. Dzięki temu kolejne punkty pracują już na jawnie zdefiniowanej bazie scoringowej, a nie na luźnych plikach tymczasowych. Rozdzielenie tabel wejściowych i wyjściowych porządkuje przepływ danych między budową wsadu inferencyjnego a zapisaniem wyników modelu. Obecność kluczy głównych i zapisanych artefaktów kontraktowych wzmacnia spójność procesu, umożliwia jednoznaczne śledzenie rekordów scoringowych oraz przygotowuje grunt pod dalszą traceability predykcji.

**Wniosek:** Dział `4.1` został domknięty poprawnie. BLOK 4 dysponuje gotową lokalną bazą scoringową, kompletnym pakietem wejściowym modelu, zdefiniowanym kontraktem tabel oraz zapisanymi artefaktami dokumentującymi warstwę bazy. Środowisko jest gotowe do przejścia do działu `4.2`, czyli pobrania modelu i konfiguracji inferencyjnej.

## 4.2 Pobieranie modelu i konfiguracji inferencyjnej

In [5]:
# 4.2.1 Definicja i walidacja pakietu modelowego

from pathlib import Path
import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_1_input_model_package_dir",
    "step4u_1_outputs_dir",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define input model package paths
step4u_2_model_file_path = Path(step4u_1_input_model_package_dir) / "b4_15_best_model.joblib"
step4u_2_inference_config_path = Path(step4u_1_input_model_package_dir) / "b4_15_inference_config.json"
step4u_2_model_registry_path = Path(step4u_1_input_model_package_dir) / "b4_15_model_registry.json"
step4u_2_handoff_manifest_path = Path(step4u_1_input_model_package_dir) / "b4_15_handoff_manifest.json"
step4u_2_feature_importance_path = Path(step4u_1_input_model_package_dir) / "b4_15_feature_importance.parquet"

# Define Section 4.2 artifacts
step4u_2_model_load_check_artifact_path = Path(step4u_1_outputs_dir) / "b4u_02_model_load_check.parquet"
step4u_2_model_load_contract_artifact_path = Path(step4u_1_outputs_dir) / "b4u_02_model_load_contract.json"

# Build model package inventory
step4u_2_model_package_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4_15_best_model.joblib",
            "artifact_role": "trained_model",
            "artifact_path": str(step4u_2_model_file_path),
            "exists": int(step4u_2_model_file_path.exists()),
            "is_required": 1,
        },
        {
            "artifact_name": "b4_15_inference_config.json",
            "artifact_role": "inference_config",
            "artifact_path": str(step4u_2_inference_config_path),
            "exists": int(step4u_2_inference_config_path.exists()),
            "is_required": 1,
        },
        {
            "artifact_name": "b4_15_model_registry.json",
            "artifact_role": "model_registry",
            "artifact_path": str(step4u_2_model_registry_path),
            "exists": int(step4u_2_model_registry_path.exists()),
            "is_required": 1,
        },
        {
            "artifact_name": "b4_15_handoff_manifest.json",
            "artifact_role": "handoff_manifest",
            "artifact_path": str(step4u_2_handoff_manifest_path),
            "exists": int(step4u_2_handoff_manifest_path.exists()),
            "is_required": 0,
        },
        {
            "artifact_name": "b4_15_feature_importance.parquet",
            "artifact_role": "feature_importance",
            "artifact_path": str(step4u_2_feature_importance_path),
            "exists": int(step4u_2_feature_importance_path.exists()),
            "is_required": 0,
        },
    ]
)

required_missing_artifacts = step4u_2_model_package_df.loc[
    (step4u_2_model_package_df["is_required"] == 1)
    & (step4u_2_model_package_df["exists"] == 0),
    "artifact_name",
].tolist()

assert not required_missing_artifacts, (
    f"Missing required model package artifacts: {required_missing_artifacts}"
)

step4u_2_model_package_summary_df = pd.DataFrame(
    [
        {
            "required_artifact_count": int((step4u_2_model_package_df["is_required"] == 1).sum()),
            "required_artifact_present_count": int(
                ((step4u_2_model_package_df["is_required"] == 1) & (step4u_2_model_package_df["exists"] == 1)).sum()
            ),
            "optional_artifact_present_count": int(
                ((step4u_2_model_package_df["is_required"] == 0) & (step4u_2_model_package_df["exists"] == 1)).sum()
            ),
            "model_package_ready": int(len(required_missing_artifacts) == 0),
        }
    ]
)

display(step4u_2_model_package_df)
display(step4u_2_model_package_summary_df)

,artifact_name,artifact_role,artifact_path,exists,is_required
0,b4_15_best_model.joblib,trained_model,/root/projects/6_samodzielny_projekt_koncowy_w...,1,1
1,b4_15_inference_config.json,inference_config,/root/projects/6_samodzielny_projekt_koncowy_w...,1,1
2,b4_15_model_registry.json,model_registry,/root/projects/6_samodzielny_projekt_koncowy_w...,1,1
3,b4_15_handoff_manifest.json,handoff_manifest,/root/projects/6_samodzielny_projekt_koncowy_w...,1,0
4,b4_15_feature_importance.parquet,feature_importance,/root/projects/6_samodzielny_projekt_koncowy_w...,1,0


,required_artifact_count,required_artifact_present_count,optional_artifact_present_count,model_package_ready
0,3,3,2,1


In [6]:
# 4.2.2 Wczytanie kontraktu inferencyjnego i metadanych modelu

import json
from pathlib import Path
import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_2_model_file_path",
    "step4u_2_inference_config_path",
    "step4u_2_model_registry_path",
    "step4u_2_handoff_manifest_path",
    "step4u_2_model_package_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define JSON reader
def read_json_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

# Load payloads
step4u_22_inference_config_payload = read_json_file(Path(step4u_2_inference_config_path))
step4u_22_model_registry_payload = read_json_file(Path(step4u_2_model_registry_path))

step4u_22_handoff_manifest_payload = None
if Path(step4u_2_handoff_manifest_path).exists():
    step4u_22_handoff_manifest_payload = read_json_file(Path(step4u_2_handoff_manifest_path))

# Load inference contract from inference_config
step4u_22_model_name = str(step4u_22_inference_config_payload["model_name"])
step4u_22_calibration_method = str(step4u_22_inference_config_payload["calibration_method"])
step4u_22_decision_threshold = float(step4u_22_inference_config_payload["decision_threshold"])
step4u_22_model_artifact_path_from_config = str(step4u_22_inference_config_payload["model_artifact_path"])
step4u_22_feature_count = int(step4u_22_inference_config_payload["feature_count"])
step4u_22_feature_list = [
    str(feature_name)
    for feature_name in step4u_22_inference_config_payload["input_feature_names"]
]

assert len(step4u_22_feature_list) > 0, "Input feature list is empty."
assert len(step4u_22_feature_list) == step4u_22_feature_count, (
    "Feature count does not match input feature list length."
)
assert 0.0 <= step4u_22_decision_threshold <= 1.0, (
    "Decision threshold must be between 0 and 1."
)

# Load registry metadata
step4u_22_registry_selected_model = step4u_22_model_registry_payload["selected_model"]
step4u_22_registry_model_name = str(step4u_22_registry_selected_model["model_name"])
step4u_22_registry_calibration_method = str(step4u_22_registry_selected_model["calibration_method"])
step4u_22_registry_decision_threshold = float(step4u_22_registry_selected_model["decision_threshold"])
step4u_22_registry_model_artifact_name = str(step4u_22_registry_selected_model["model_artifact_name"])
step4u_22_registry_model_artifact_path = str(step4u_22_registry_selected_model["model_artifact_path"])

# Load optional handoff metadata
step4u_22_handoff_model_name = None
step4u_22_handoff_calibration_method = None
step4u_22_handoff_decision_threshold = None

if step4u_22_handoff_manifest_payload is not None:
    step4u_22_handoff_selected_model = step4u_22_handoff_manifest_payload["selected_model"]
    step4u_22_handoff_model_name = str(step4u_22_handoff_selected_model["model_name"])
    step4u_22_handoff_calibration_method = str(step4u_22_handoff_selected_model["calibration_method"])
    step4u_22_handoff_decision_threshold = float(step4u_22_handoff_selected_model["decision_threshold"])

# Validate consistency across package metadata
assert step4u_22_model_name == step4u_22_registry_model_name, (
    "Model name is not consistent between inference_config and model_registry."
)
assert step4u_22_calibration_method == step4u_22_registry_calibration_method, (
    "Calibration method is not consistent between inference_config and model_registry."
)
assert step4u_22_decision_threshold == step4u_22_registry_decision_threshold, (
    "Decision threshold is not consistent between inference_config and model_registry."
)
assert Path(step4u_2_model_file_path).name == step4u_22_registry_model_artifact_name, (
    "Model artifact name is not consistent with model_registry."
)

if step4u_22_handoff_manifest_payload is not None:
    assert step4u_22_model_name == step4u_22_handoff_model_name, (
        "Model name is not consistent between inference_config and handoff_manifest."
    )
    assert step4u_22_calibration_method == step4u_22_handoff_calibration_method, (
        "Calibration method is not consistent between inference_config and handoff_manifest."
    )
    assert step4u_22_decision_threshold == step4u_22_handoff_decision_threshold, (
        "Decision threshold is not consistent between inference_config and handoff_manifest."
    )

# Define model release tag from artifact naming
step4u_22_model_artifact_name = Path(step4u_2_model_file_path).name
step4u_22_model_release_tag = step4u_22_model_artifact_name.replace("_best_model.joblib", "")

assert step4u_22_model_release_tag.startswith("b4_"), (
    "Model release tag does not match expected package naming."
)

# Build payload summary
step4u_22_payload_summary_df = pd.DataFrame(
    [
        {
            "payload_name": "inference_config",
            "payload_path": str(step4u_2_inference_config_path),
            "exists": int(Path(step4u_2_inference_config_path).exists()),
            "top_level_key_count": int(len(step4u_22_inference_config_payload)),
        },
        {
            "payload_name": "model_registry",
            "payload_path": str(step4u_2_model_registry_path),
            "exists": int(Path(step4u_2_model_registry_path).exists()),
            "top_level_key_count": int(len(step4u_22_model_registry_payload)),
        },
        {
            "payload_name": "handoff_manifest",
            "payload_path": str(step4u_2_handoff_manifest_path),
            "exists": int(Path(step4u_2_handoff_manifest_path).exists()),
            "top_level_key_count": int(
                len(step4u_22_handoff_manifest_payload)
                if isinstance(step4u_22_handoff_manifest_payload, dict)
                else 0
            ),
        },
    ]
)

# Build inference contract summary
step4u_22_inference_contract_summary_df = pd.DataFrame(
    [
        {
            "model_name": step4u_22_model_name,
            "model_release_tag": step4u_22_model_release_tag,
            "calibration_method": step4u_22_calibration_method,
            "decision_threshold": step4u_22_decision_threshold,
            "feature_count": step4u_22_feature_count,
            "model_artifact_name": step4u_22_model_artifact_name,
            "model_file_exists": int(Path(step4u_2_model_file_path).exists()),
            "contract_consistent": 1,
            "ready_for_scoring": int(
                Path(step4u_2_model_file_path).exists()
                and len(step4u_22_feature_list) == step4u_22_feature_count
                and 0.0 <= step4u_22_decision_threshold <= 1.0
            ),
        }
    ]
)

# Build feature list preview
step4u_22_feature_list_df = pd.DataFrame(
    {
        "feature_position": list(range(step4u_22_feature_count)),
        "feature_name": step4u_22_feature_list,
    }
)

display(step4u_22_payload_summary_df)
display(step4u_22_inference_contract_summary_df)
display(step4u_22_feature_list_df.head(20))

,payload_name,payload_path,exists,top_level_key_count
0,inference_config,/root/projects/6_samodzielny_projekt_koncowy_w...,1,9
1,model_registry,/root/projects/6_samodzielny_projekt_koncowy_w...,1,10
2,handoff_manifest,/root/projects/6_samodzielny_projekt_koncowy_w...,1,11


,model_name,model_release_tag,calibration_method,decision_threshold,feature_count,model_artifact_name,model_file_exists,contract_consistent,ready_for_scoring
0,lightgbm_tuned,b4_15,raw,0.55,34,b4_15_best_model.joblib,1,1,1


,feature_position,feature_name
0,0,alert_hours_roll_sum_14
1,1,alert_hours_lag_1
2,2,alert_severity_roll_max_7
3,3,consecutive_alert_days_before_t
4,4,alert_lag_1
5,5,alert_lag_2
6,6,alert_lag_3
7,7,alert_lag_7
8,8,deficit_alert_lag_1
9,9,high_severity_alert_last_7d


In [7]:
# 4.2.3 Wczytanie obiektu modelowego

from pathlib import Path
import pandas as pd
import joblib

# Validate required objects from previous steps
required_objects = [
    "step4u_2_model_file_path",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_decision_threshold",
    "step4u_22_feature_count",
    "step4u_22_feature_list",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Load trained model object
step4u_23_model_file_path = Path(step4u_2_model_file_path)
assert step4u_23_model_file_path.exists(), f"Model file does not exist: {step4u_23_model_file_path}"

step4u_23_model_object = joblib.load(step4u_23_model_file_path)

# Read model object metadata
step4u_23_model_class_name = type(step4u_23_model_object).__name__
step4u_23_model_module_name = type(step4u_23_model_object).__module__

step4u_23_has_predict = int(hasattr(step4u_23_model_object, "predict"))
step4u_23_has_predict_proba = int(hasattr(step4u_23_model_object, "predict_proba"))
step4u_23_has_feature_names_in = int(hasattr(step4u_23_model_object, "feature_names_in_"))
step4u_23_has_n_features_in = int(hasattr(step4u_23_model_object, "n_features_in_"))
step4u_23_has_classes = int(hasattr(step4u_23_model_object, "classes_"))

step4u_23_model_feature_count_from_object = (
    int(step4u_23_model_object.n_features_in_)
    if step4u_23_has_n_features_in == 1
    else pd.NA
)

step4u_23_model_classes = (
    [str(class_value) for class_value in step4u_23_model_object.classes_.tolist()]
    if step4u_23_has_classes == 1
    else []
)

step4u_23_model_feature_names_from_object = (
    [str(feature_name) for feature_name in step4u_23_model_object.feature_names_in_.tolist()]
    if step4u_23_has_feature_names_in == 1
    else []
)

# Validate inference readiness
assert step4u_23_has_predict == 1, "Loaded model object does not expose predict."
assert step4u_23_has_predict_proba == 1, "Loaded model object does not expose predict_proba."

if step4u_23_has_n_features_in == 1:
    assert step4u_23_model_feature_count_from_object == step4u_22_feature_count, (
        "Model feature count is not consistent with inference contract."
    )

if step4u_23_has_feature_names_in == 1:
    assert len(step4u_23_model_feature_names_from_object) == step4u_22_feature_count, (
        "Model feature names length is not consistent with inference contract."
    )

# Build model object summary
step4u_23_model_object_summary_df = pd.DataFrame(
    [
        {
            "model_name": step4u_22_model_name,
            "model_release_tag": step4u_22_model_release_tag,
            "model_class_name": step4u_23_model_class_name,
            "model_module_name": step4u_23_model_module_name,
            "decision_threshold": step4u_22_decision_threshold,
            "feature_count_from_contract": int(step4u_22_feature_count),
            "feature_count_from_object": step4u_23_model_feature_count_from_object,
            "class_count": int(len(step4u_23_model_classes)),
            "has_predict": step4u_23_has_predict,
            "has_predict_proba": step4u_23_has_predict_proba,
            "has_feature_names_in": step4u_23_has_feature_names_in,
            "contract_consistent": 1,
            "model_ready_for_inference": 1,
        }
    ]
)

# Build model class preview
step4u_23_model_classes_df = pd.DataFrame(
    {
        "class_position": list(range(len(step4u_23_model_classes))),
        "class_label": step4u_23_model_classes,
    }
)

# Build model feature name preview
step4u_23_model_feature_names_df = pd.DataFrame(
    {
        "feature_position": list(range(len(step4u_23_model_feature_names_from_object))),
        "feature_name": step4u_23_model_feature_names_from_object,
    }
)

display(step4u_23_model_object_summary_df)
display(step4u_23_model_classes_df)
display(step4u_23_model_feature_names_df.head(20))

,model_name,model_release_tag,model_class_name,model_module_name,decision_threshold,feature_count_from_contract,feature_count_from_object,class_count,has_predict,has_predict_proba,has_feature_names_in,contract_consistent,model_ready_for_inference
0,lightgbm_tuned,b4_15,LGBMClassifier,lightgbm.sklearn,0.55,34,34,2,1,1,1,1,1


,class_position,class_label
0,0,0
1,1,1


,feature_position,feature_name
0,0,alert_hours_roll_sum_14
1,1,alert_hours_lag_1
2,2,alert_severity_roll_max_7
3,3,consecutive_alert_days_before_t
4,4,alert_lag_1
5,5,alert_lag_2
6,6,alert_lag_3
7,7,alert_lag_7
8,8,deficit_alert_lag_1
9,9,high_severity_alert_last_7d


In [8]:
# 4.2.4 Zapis artefaktów sekcji 4.2

import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_2_model_load_check_artifact_path",
    "step4u_2_model_load_contract_artifact_path",
    "step4u_2_model_file_path",
    "step4u_2_inference_config_path",
    "step4u_2_model_registry_path",
    "step4u_2_handoff_manifest_path",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_calibration_method",
    "step4u_22_decision_threshold",
    "step4u_22_feature_count",
    "step4u_22_feature_list",
    "step4u_22_payload_summary_df",
    "step4u_22_inference_contract_summary_df",
    "step4u_23_model_object_summary_df",
    "step4u_23_model_classes_df",
    "step4u_23_model_feature_names_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

step4u_24_model_load_check_artifact_path = Path(step4u_2_model_load_check_artifact_path)
step4u_24_model_load_contract_artifact_path = Path(step4u_2_model_load_contract_artifact_path)

step4u_24_model_load_check_artifact_path.parent.mkdir(parents=True, exist_ok=True)
step4u_24_model_load_contract_artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Build unified model load check artifact
step4u_24_check_rows = []

for _, row in step4u_22_payload_summary_df.iterrows():
    step4u_24_check_rows.append(
        {
            "check_section": "payloads",
            "item_name": row["payload_name"],
            "item_type": "json_payload",
            "status": int(row["exists"]),
            "detail_json": json.dumps(
                {
                    "payload_path": row["payload_path"],
                    "exists": int(row["exists"]),
                    "top_level_key_count": int(row["top_level_key_count"]),
                },
                ensure_ascii=False,
            ),
        }
    )

step4u_24_inference_contract_row = step4u_22_inference_contract_summary_df.iloc[0]
step4u_24_check_rows.append(
    {
        "check_section": "inference_contract",
        "item_name": step4u_24_inference_contract_row["model_name"],
        "item_type": "contract_summary",
        "status": int(step4u_24_inference_contract_row["ready_for_scoring"]),
        "detail_json": json.dumps(
            {
                "model_release_tag": step4u_24_inference_contract_row["model_release_tag"],
                "calibration_method": step4u_24_inference_contract_row["calibration_method"],
                "decision_threshold": float(step4u_24_inference_contract_row["decision_threshold"]),
                "feature_count": int(step4u_24_inference_contract_row["feature_count"]),
                "model_artifact_name": step4u_24_inference_contract_row["model_artifact_name"],
                "contract_consistent": int(step4u_24_inference_contract_row["contract_consistent"]),
                "ready_for_scoring": int(step4u_24_inference_contract_row["ready_for_scoring"]),
            },
            ensure_ascii=False,
        ),
    }
)

step4u_24_model_object_row = step4u_23_model_object_summary_df.iloc[0]
step4u_24_check_rows.append(
    {
        "check_section": "model_object",
        "item_name": step4u_24_model_object_row["model_class_name"],
        "item_type": "model_summary",
        "status": int(step4u_24_model_object_row["model_ready_for_inference"]),
        "detail_json": json.dumps(
            {
                "model_name": step4u_24_model_object_row["model_name"],
                "model_release_tag": step4u_24_model_object_row["model_release_tag"],
                "model_module_name": step4u_24_model_object_row["model_module_name"],
                "decision_threshold": float(step4u_24_model_object_row["decision_threshold"]),
                "feature_count_from_contract": int(step4u_24_model_object_row["feature_count_from_contract"]),
                "feature_count_from_object": int(step4u_24_model_object_row["feature_count_from_object"]),
                "class_count": int(step4u_24_model_object_row["class_count"]),
                "has_predict": int(step4u_24_model_object_row["has_predict"]),
                "has_predict_proba": int(step4u_24_model_object_row["has_predict_proba"]),
                "has_feature_names_in": int(step4u_24_model_object_row["has_feature_names_in"]),
                "contract_consistent": int(step4u_24_model_object_row["contract_consistent"]),
                "model_ready_for_inference": int(step4u_24_model_object_row["model_ready_for_inference"]),
            },
            ensure_ascii=False,
        ),
    }
)

for _, row in step4u_23_model_classes_df.iterrows():
    step4u_24_check_rows.append(
        {
            "check_section": "model_classes",
            "item_name": str(row["class_label"]),
            "item_type": "class_label",
            "status": 1,
            "detail_json": json.dumps(
                {
                    "class_position": int(row["class_position"]),
                    "class_label": str(row["class_label"]),
                },
                ensure_ascii=False,
            ),
        }
    )

for _, row in step4u_23_model_feature_names_df.iterrows():
    step4u_24_check_rows.append(
        {
            "check_section": "model_features",
            "item_name": row["feature_name"],
            "item_type": "feature_name",
            "status": 1,
            "detail_json": json.dumps(
                {
                    "feature_position": int(row["feature_position"]),
                    "feature_name": row["feature_name"],
                },
                ensure_ascii=False,
            ),
        }
    )

step4u_24_model_load_check_df = pd.DataFrame(step4u_24_check_rows)

step4u_24_model_load_check_df["check_section"] = step4u_24_model_load_check_df["check_section"].astype("string")
step4u_24_model_load_check_df["item_name"] = step4u_24_model_load_check_df["item_name"].astype("string")
step4u_24_model_load_check_df["item_type"] = step4u_24_model_load_check_df["item_type"].astype("string")
step4u_24_model_load_check_df["status"] = step4u_24_model_load_check_df["status"].astype("int8")
step4u_24_model_load_check_df["detail_json"] = step4u_24_model_load_check_df["detail_json"].astype("string")

step4u_24_model_load_check_table = pa.Table.from_pandas(
    step4u_24_model_load_check_df,
    preserve_index=False,
)
pq.write_table(
    step4u_24_model_load_check_table,
    step4u_24_model_load_check_artifact_path,
)

# Build model load contract artifact
step4u_24_model_load_contract_payload = {
    "section_name": "4.2",
    "model_name": step4u_22_model_name,
    "model_release_tag": step4u_22_model_release_tag,
    "calibration_method": step4u_22_calibration_method,
    "decision_threshold": float(step4u_22_decision_threshold),
    "feature_count": int(step4u_22_feature_count),
    "feature_list": step4u_22_feature_list,
    "model_artifact_path": str(step4u_2_model_file_path),
    "inference_config_path": str(step4u_2_inference_config_path),
    "model_registry_path": str(step4u_2_model_registry_path),
    "handoff_manifest_path": str(step4u_2_handoff_manifest_path),
    "payload_summary": step4u_22_payload_summary_df.to_dict(orient="records"),
    "inference_contract_summary": step4u_22_inference_contract_summary_df.to_dict(orient="records"),
    "model_object_summary": step4u_23_model_object_summary_df.to_dict(orient="records"),
    "model_classes": step4u_23_model_classes_df.to_dict(orient="records"),
    "model_feature_names": step4u_23_model_feature_names_df.to_dict(orient="records"),
}

with open(step4u_24_model_load_contract_artifact_path, "w", encoding="utf-8") as file:
    json.dump(step4u_24_model_load_contract_payload, file, ensure_ascii=False, indent=2)

step4u_24_artifact_save_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4u_02_model_load_check.parquet",
            "artifact_path": str(step4u_24_model_load_check_artifact_path),
            "exists": int(step4u_24_model_load_check_artifact_path.exists()),
            "row_count": int(step4u_24_model_load_check_df.shape[0]),
        },
        {
            "artifact_name": "b4u_02_model_load_contract.json",
            "artifact_path": str(step4u_24_model_load_contract_artifact_path),
            "exists": int(step4u_24_model_load_contract_artifact_path.exists()),
            "row_count": pd.NA,
        },
    ]
)

display(step4u_24_artifact_save_df)

,artifact_name,artifact_path,exists,row_count
0,b4u_02_model_load_check.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1,41
1,b4u_02_model_load_contract.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1,<NA>


## Podsumowanie działu 4.2 Pobieranie modelu i konfiguracji inferencyjnej

**Kontekst inżynieryjny:** W dziale `4.2` wczytano i zinwentaryzowano pakiet modelowy z katalogu `input_model_package`. Najpierw potwierdzono obecność plików `b4_15_best_model.joblib`, `b4_15_inference_config.json` oraz `b4_15_model_registry.json` jako artefaktów wymaganych, a także `b4_15_handoff_manifest.json` i `b4_15_feature_importance.parquet` jako artefaktów wspierających. Następnie odczytano kontrakt inferencyjny i metadane modelu z payloadów JSON, obejmujące nazwę modelu, metodę kalibracji, próg decyzyjny, liczbę cech oraz listę cech wejściowych. W kolejnym kroku wczytano sam obiekt modelowy z pliku `b4_15_best_model.joblib` i odczytano jego właściwości operacyjne, w tym klasę modelu, moduł, liczbę klas oraz zgodność liczby cech z kontraktem inferencyjnym. Na końcu zapisano artefakty sekcji `4.2`: `b4u_02_model_load_check.parquet` oraz `b4u_02_model_load_contract.json`.

**Interpretacja wyniku:** Pakiet modelowy jest kompletny i gotowy do użycia w BLOKU 4. Wszystkie `3` wymagane artefakty wejściowe są obecne, a dodatkowo dostępne są `2` artefakty wspierające. Kontrakt inferencyjny wskazuje model `lightgbm_tuned` z pakietu `b4_15`, metodę `raw`, próg decyzyjny `0.55` oraz `34` cechy wejściowe. Lista cech została wczytana poprawnie i ma ustaloną kolejność. Załadowany obiekt modelowy ma klasę `LGBMClassifier`, udostępnia metody `predict` i `predict_proba`, przechowuje nazwy cech wejściowych oraz operuje na `2` klasach binarnych `0` i `1`. Liczba cech odczytana z kontraktu i z obiektu modelowego wynosi `34`, co potwierdza spójność warstwy inferencyjnej. Zapisany artefakt `b4u_02_model_load_check.parquet` zawiera `41` rekordów obejmujących payloady, kontrakt inferencyjny, model, klasy oraz cechy.

**Znaczenie biznesowe:** Dział `4.2` formalizuje, jaki dokładnie model będzie używany w dalszym scoringu dziennym. Dzięki temu kolejne etapy BLOKU 4 pracują już na jawnie zdefiniowanym modelu, ustalonym progu i stałej liście cech, bez zmiany logiki modelowania. Uporządkowanie tych elementów wzmacnia przewidywalność procesu inferencyjnego, ułatwia kontrolę zgodności wsadu wejściowego z kontraktem modelu oraz przygotowuje podstawę pod późniejszą traceability wyników predykcyjnych.

**Wniosek:** Dział `4.2` został domknięty poprawnie. BLOK 4 dysponuje kompletnym pakietem modelowym, wczytanym kontraktem inferencyjnym, załadowanym obiektem modelowym oraz zapisanymi artefaktami dokumentującymi warstwę modelu i konfiguracji. Środowisko jest gotowe do przejścia do działu `4.3`, czyli przygotowania wsadu danych do predykcji na kolejny dzień.

## 4.3 Przygotowanie wsadu danych do predykcji na kolejny dzień

In [9]:
# 4.3.1 Definicja źródeł wsadu inferencyjnego i trybu wyboru daty scoringowej

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_1_project_root",
    "step4u_1_outputs_dir",
    "step4u_22_feature_list",
    "step4u_22_feature_count",
    "step4u_22_decision_threshold",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define Section 4.3 artifacts
step4u_31_scoring_input_artifact_path = Path(step4u_1_outputs_dir) / "b4u_03_scoring_input.parquet"
step4u_31_scoring_input_contract_artifact_path = Path(step4u_1_outputs_dir) / "b4u_03_scoring_input_contract.json"

# Define source artifact names
step4u_31_model_ready_artifact_name = "b3_13_model_ready_dataset.parquet"
step4u_31_null_semantics_artifact_name = "b3_13_null_semantics.parquet"

# Define scoring date selection mode
step4u_31_scoring_date_mode = "latest_available"
step4u_31_selected_scoring_date_input = None

allowed_scoring_date_modes = {"latest_available", "selected_date"}
assert step4u_31_scoring_date_mode in allowed_scoring_date_modes, (
    f"Unsupported scoring date mode: {step4u_31_scoring_date_mode}"
)

# Define search roots
step4u_31_search_roots = [
    Path(step4u_1_project_root),
    Path(step4u_1_project_root).parent,
]

# Define exact file finder
def find_first_file_by_name(search_roots: list[Path], file_name: str) -> Path | None:
    for root in search_roots:
        direct_path = root / file_name
        if direct_path.exists():
            return direct_path.resolve()

        matches = sorted(root.rglob(file_name))
        if matches:
            return matches[0].resolve()

    return None

# Resolve source artifact paths
step4u_31_model_ready_path = find_first_file_by_name(
    step4u_31_search_roots,
    step4u_31_model_ready_artifact_name,
)
step4u_31_null_semantics_path = find_first_file_by_name(
    step4u_31_search_roots,
    step4u_31_null_semantics_artifact_name,
)

assert step4u_31_model_ready_path is not None, (
    f"Source artifact was not found: {step4u_31_model_ready_artifact_name}"
)
assert step4u_31_null_semantics_path is not None, (
    f"Source artifact was not found: {step4u_31_null_semantics_artifact_name}"
)

# Read parquet metadata and schemas
step4u_31_model_ready_parquet = pq.ParquetFile(step4u_31_model_ready_path)
step4u_31_null_semantics_parquet = pq.ParquetFile(step4u_31_null_semantics_path)

step4u_31_model_ready_columns = list(step4u_31_model_ready_parquet.schema.names)
step4u_31_null_semantics_columns = list(step4u_31_null_semantics_parquet.schema.names)

# Validate feature contract against source schema
step4u_31_missing_contract_features = [
    feature_name
    for feature_name in step4u_22_feature_list
    if feature_name not in step4u_31_model_ready_columns
]

step4u_31_available_contract_features = [
    feature_name
    for feature_name in step4u_22_feature_list
    if feature_name in step4u_31_model_ready_columns
]

assert not step4u_31_missing_contract_features, (
    f"Missing required features in model-ready dataset: {step4u_31_missing_contract_features}"
)

# Validate minimum key columns
step4u_31_required_key_columns = ["activity_date", "station_id"]

step4u_31_missing_key_columns = [
    column_name
    for column_name in step4u_31_required_key_columns
    if column_name not in step4u_31_model_ready_columns
]

assert not step4u_31_missing_key_columns, (
    f"Missing required key columns in model-ready dataset: {step4u_31_missing_key_columns}"
)

# Build source artifact inventory
step4u_31_source_artifact_df = pd.DataFrame(
    [
        {
            "artifact_name": step4u_31_model_ready_artifact_name,
            "artifact_role": "model_ready_source_dataset",
            "artifact_path": str(step4u_31_model_ready_path),
            "exists": 1,
            "row_count": int(step4u_31_model_ready_parquet.metadata.num_rows),
            "column_count": int(len(step4u_31_model_ready_columns)),
        },
        {
            "artifact_name": step4u_31_null_semantics_artifact_name,
            "artifact_role": "null_semantics_contract",
            "artifact_path": str(step4u_31_null_semantics_path),
            "exists": 1,
            "row_count": int(step4u_31_null_semantics_parquet.metadata.num_rows),
            "column_count": int(len(step4u_31_null_semantics_columns)),
        },
    ]
)

# Build scoring date selection summary
step4u_31_scoring_date_mode_df = pd.DataFrame(
    [
        {
            "scoring_date_mode": step4u_31_scoring_date_mode,
            "selected_scoring_date_input": step4u_31_selected_scoring_date_input,
            "required_key_column_count": int(len(step4u_31_required_key_columns)),
            "expected_feature_count": int(step4u_22_feature_count),
            "available_feature_count": int(len(step4u_31_available_contract_features)),
            "missing_feature_count": int(len(step4u_31_missing_contract_features)),
            "decision_threshold": float(step4u_22_decision_threshold),
            "contract_ready": int(len(step4u_31_missing_contract_features) == 0),
        }
    ]
)

display(step4u_31_source_artifact_df)
display(step4u_31_scoring_date_mode_df)

,artifact_name,artifact_role,artifact_path,exists,row_count,column_count
0,b3_13_model_ready_dataset.parquet,model_ready_source_dataset,/root/projects/6_samodzielny_projekt_koncowy_w...,1,218222,38
1,b3_13_null_semantics.parquet,null_semantics_contract,/root/projects/6_samodzielny_projekt_koncowy_w...,1,34,17


,scoring_date_mode,selected_scoring_date_input,required_key_column_count,expected_feature_count,available_feature_count,missing_feature_count,decision_threshold,contract_ready
0,latest_available,None,2,34,34,0,0.55,1


In [10]:
# 4.3.2 Ustalenie finalnej daty scoringu

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_31_model_ready_path",
    "step4u_31_scoring_date_mode",
    "step4u_31_selected_scoring_date_input",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Load available scoring dates
step4u_32_available_dates_table = pq.read_table(
    Path(step4u_31_model_ready_path),
    columns=["activity_date"],
)
step4u_32_available_dates_df = step4u_32_available_dates_table.to_pandas()

step4u_32_available_dates_df["activity_date"] = pd.to_datetime(
    step4u_32_available_dates_df["activity_date"],
    errors="coerce",
).dt.normalize()

assert step4u_32_available_dates_df["activity_date"].notna().all(), (
    "Column activity_date contains invalid values."
)

step4u_32_available_dates_df = (
    step4u_32_available_dates_df[["activity_date"]]
    .drop_duplicates()
    .sort_values("activity_date")
    .reset_index(drop=True)
)

assert not step4u_32_available_dates_df.empty, "No available scoring dates were found."

# Resolve final scoring date
if step4u_31_scoring_date_mode == "latest_available":
    step4u_32_selected_scoring_date = step4u_32_available_dates_df["activity_date"].max()
else:
    assert step4u_31_selected_scoring_date_input is not None, (
        "Selected scoring date input is required for selected_date mode."
    )

    step4u_32_selected_scoring_date = pd.to_datetime(
        step4u_31_selected_scoring_date_input,
        errors="coerce",
    )
    assert pd.notna(step4u_32_selected_scoring_date), (
        "Selected scoring date input is invalid."
    )

    step4u_32_selected_scoring_date = step4u_32_selected_scoring_date.normalize()

    assert step4u_32_selected_scoring_date in step4u_32_available_dates_df["activity_date"].tolist(), (
        f"Selected scoring date is not available: {step4u_32_selected_scoring_date.date()}"
    )

# Build scoring date summary
step4u_32_scoring_date_summary_df = pd.DataFrame(
    [
        {
            "scoring_date_mode": step4u_31_scoring_date_mode,
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "min_available_scoring_date": step4u_32_available_dates_df["activity_date"].min(),
            "max_available_scoring_date": step4u_32_available_dates_df["activity_date"].max(),
            "available_scoring_date_count": int(step4u_32_available_dates_df.shape[0]),
            "scoring_date_ready": 1,
        }
    ]
)

display(step4u_32_scoring_date_summary_df)
display(step4u_32_available_dates_df.tail(10))

,scoring_date_mode,selected_scoring_date,min_available_scoring_date,max_available_scoring_date,available_scoring_date_count,scoring_date_ready
0,latest_available,2020-10-31,2017-05-02,2020-10-31,826,1


,activity_date
816,2020-10-22
817,2020-10-23
818,2020-10-24
819,2020-10-25
820,2020-10-26
821,2020-10-27
822,2020-10-28
823,2020-10-29
824,2020-10-30
825,2020-10-31


In [11]:
# 4.3.3 Wczytanie batcha scoringowego dla wybranej daty

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_31_model_ready_path",
    "step4u_31_null_semantics_path",
    "step4u_31_model_ready_columns",
    "step4u_22_feature_list",
    "step4u_22_feature_count",
    "step4u_32_selected_scoring_date",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define required key columns
step4u_33_required_key_columns = ["activity_date", "station_id"]

missing_key_columns = [
    column_name
    for column_name in step4u_33_required_key_columns
    if column_name not in step4u_31_model_ready_columns
]
assert not missing_key_columns, f"Missing required key columns: {missing_key_columns}"

# Define optional non-model columns
step4u_33_optional_control_columns = [
    column_name
    for column_name in ["split_name"]
    if column_name in step4u_31_model_ready_columns
]

step4u_33_optional_context_candidates = [
    "hub_flag",
    "is_cold_start",
    "is_holiday",
    "is_business_free_day",
    "alert_hours_roll_sum_14",
    "alert_hours_lag_1",
    "consecutive_alert_days_before_t",
]

step4u_33_optional_context_columns = [
    column_name
    for column_name in step4u_33_optional_context_candidates
    if column_name in step4u_31_model_ready_columns
]

# Define source columns with stable order
step4u_33_source_columns = list(
    dict.fromkeys(
        step4u_33_required_key_columns
        + step4u_33_optional_control_columns
        + step4u_33_optional_context_columns
        + step4u_22_feature_list
    )
)

# Load source data for the selected scoring date
step4u_33_source_table = pq.read_table(
    Path(step4u_31_model_ready_path),
    columns=step4u_33_source_columns,
)

step4u_33_source_df = step4u_33_source_table.to_pandas()

step4u_33_source_df["activity_date"] = pd.to_datetime(
    step4u_33_source_df["activity_date"],
    errors="coerce",
).dt.normalize()

step4u_33_source_df["station_id"] = step4u_33_source_df["station_id"].astype("string")

assert step4u_33_source_df["activity_date"].notna().all(), (
    "Column activity_date contains invalid values."
)
assert step4u_33_source_df["station_id"].notna().all(), (
    "Column station_id contains missing values."
)

step4u_33_scoring_batch_source_df = (
    step4u_33_source_df.loc[
        step4u_33_source_df["activity_date"] == step4u_32_selected_scoring_date
    ]
    .sort_values(["activity_date", "station_id"])
    .reset_index(drop=True)
)

assert not step4u_33_scoring_batch_source_df.empty, "Selected scoring batch is empty."

step4u_33_duplicate_key_count = int(
    step4u_33_scoring_batch_source_df.duplicated(
        subset=["activity_date", "station_id"]
    ).sum()
)
assert step4u_33_duplicate_key_count == 0, (
    "Selected scoring batch contains duplicate activity_date and station_id keys."
)

# Load null semantics contract
step4u_33_null_semantics_table = pq.read_table(Path(step4u_31_null_semantics_path))
step4u_33_null_semantics_df = step4u_33_null_semantics_table.to_pandas().copy()

step4u_33_null_semantics_feature_key_candidates = [
    "feature_name",
    "column_name",
    "feature",
    "variable_name",
    "variable",
]

step4u_33_null_semantics_feature_key = next(
    (
        column_name
        for column_name in step4u_33_null_semantics_feature_key_candidates
        if column_name in step4u_33_null_semantics_df.columns
    ),
    None,
)

assert step4u_33_null_semantics_feature_key is not None, (
    "Null semantics feature key column was not found."
)

step4u_33_null_semantics_df[step4u_33_null_semantics_feature_key] = (
    step4u_33_null_semantics_df[step4u_33_null_semantics_feature_key]
    .astype("string")
)

step4u_33_null_semantics_feature_df = (
    step4u_33_null_semantics_df.loc[
        step4u_33_null_semantics_df[step4u_33_null_semantics_feature_key].isin(step4u_22_feature_list)
    ]
    .copy()
    .sort_values(step4u_33_null_semantics_feature_key)
    .reset_index(drop=True)
)

step4u_33_missing_null_semantics_features = [
    feature_name
    for feature_name in step4u_22_feature_list
    if feature_name
    not in step4u_33_null_semantics_feature_df[step4u_33_null_semantics_feature_key].tolist()
]

assert not step4u_33_missing_null_semantics_features, (
    f"Missing null semantics definitions: {step4u_33_missing_null_semantics_features}"
)

step4u_33_null_semantics_duplicate_count = int(
    step4u_33_null_semantics_feature_df.duplicated(
        subset=[step4u_33_null_semantics_feature_key]
    ).sum()
)
assert step4u_33_null_semantics_duplicate_count == 0, (
    "Null semantics contract contains duplicate feature definitions."
)

# Build scoring batch summary
step4u_33_scoring_batch_summary_df = pd.DataFrame(
    [
        {
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "record_count": int(step4u_33_scoring_batch_source_df.shape[0]),
            "station_count": int(step4u_33_scoring_batch_source_df["station_id"].nunique()),
            "model_feature_count": int(len(step4u_22_feature_list)),
            "non_model_column_count": int(len(step4u_33_source_columns) - len(step4u_22_feature_list)),
            "duplicate_key_count": step4u_33_duplicate_key_count,
            "scoring_batch_ready": 1,
        }
    ]
)

# Build null semantics summary
step4u_33_null_semantics_summary_df = pd.DataFrame(
    [
        {
            "feature_key_column": step4u_33_null_semantics_feature_key,
            "expected_feature_count": int(step4u_22_feature_count),
            "available_feature_count": int(step4u_33_null_semantics_feature_df.shape[0]),
            "duplicate_feature_definition_count": step4u_33_null_semantics_duplicate_count,
            "missing_feature_definition_count": int(len(step4u_33_missing_null_semantics_features)),
            "null_semantics_ready": int(len(step4u_33_missing_null_semantics_features) == 0),
        }
    ]
)

display(step4u_33_scoring_batch_summary_df)
display(step4u_33_null_semantics_summary_df)

,selected_scoring_date,record_count,station_count,model_feature_count,non_model_column_count,duplicate_key_count,scoring_batch_ready
0,2020-10-31,326,326,34,3,0,1


,feature_key_column,expected_feature_count,available_feature_count,duplicate_feature_definition_count,missing_feature_definition_count,null_semantics_ready
0,column_name,34,34,0,0,1


In [12]:
# 4.3.4 Walidacja kolejności cech i budowa macierzy wejściowej modelu

import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_22_feature_list",
    "step4u_22_feature_count",
    "step4u_23_has_feature_names_in",
    "step4u_23_model_feature_names_from_object",
    "step4u_33_scoring_batch_source_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define contract feature checks
step4u_34_contract_feature_list = [str(feature_name) for feature_name in step4u_22_feature_list]
step4u_34_contract_feature_count = int(step4u_22_feature_count)

step4u_34_contract_duplicate_feature_count = int(
    pd.Series(step4u_34_contract_feature_list).duplicated().sum()
)
assert step4u_34_contract_duplicate_feature_count == 0, (
    "Feature contract contains duplicate feature names."
)

assert len(step4u_34_contract_feature_list) == step4u_34_contract_feature_count, (
    "Feature contract length does not match expected feature count."
)

# Validate batch columns against the feature contract
step4u_34_batch_columns = [str(column_name) for column_name in step4u_33_scoring_batch_source_df.columns]

step4u_34_missing_batch_features = [
    feature_name
    for feature_name in step4u_34_contract_feature_list
    if feature_name not in step4u_34_batch_columns
]
assert not step4u_34_missing_batch_features, (
    f"Missing required feature columns in scoring batch: {step4u_34_missing_batch_features}"
)

# Build model input matrix with exact contract order
step4u_34_model_input_df = step4u_33_scoring_batch_source_df[
    step4u_34_contract_feature_list
].copy()

step4u_34_model_input_feature_list = list(step4u_34_model_input_df.columns)
step4u_34_contract_order_consistent = int(
    step4u_34_model_input_feature_list == step4u_34_contract_feature_list
)
assert step4u_34_contract_order_consistent == 1, (
    "Model input feature order is not consistent with the inference contract."
)

step4u_34_model_input_duplicate_column_count = int(
    pd.Series(step4u_34_model_input_feature_list).duplicated().sum()
)
assert step4u_34_model_input_duplicate_column_count == 0, (
    "Model input contains duplicate feature columns."
)

# Validate feature order against the loaded model object
step4u_34_model_object_order_consistent = pd.NA
step4u_34_model_object_feature_count = pd.NA

if step4u_23_has_feature_names_in == 1:
    step4u_34_model_object_feature_list = [
        str(feature_name)
        for feature_name in step4u_23_model_feature_names_from_object
    ]
    step4u_34_model_object_feature_count = int(len(step4u_34_model_object_feature_list))
    step4u_34_model_object_order_consistent = int(
        step4u_34_model_object_feature_list == step4u_34_contract_feature_list
    )
    assert step4u_34_model_object_order_consistent == 1, (
        "Model object feature order is not consistent with the inference contract."
    )
else:
    step4u_34_model_object_feature_list = []

# Build input matrix summary
step4u_34_model_input_summary_df = pd.DataFrame(
    [
        {
            "selected_scoring_date": step4u_33_scoring_batch_summary_df.loc[0, "selected_scoring_date"],
            "record_count": int(step4u_34_model_input_df.shape[0]),
            "feature_count_from_contract": step4u_34_contract_feature_count,
            "feature_count_from_input_matrix": int(step4u_34_model_input_df.shape[1]),
            "duplicate_contract_feature_count": step4u_34_contract_duplicate_feature_count,
            "duplicate_input_column_count": step4u_34_model_input_duplicate_column_count,
            "missing_batch_feature_count": int(len(step4u_34_missing_batch_features)),
            "contract_order_consistent": step4u_34_contract_order_consistent,
            "model_object_feature_count": step4u_34_model_object_feature_count,
            "model_object_order_consistent": step4u_34_model_object_order_consistent,
            "model_input_ready": 1,
        }
    ]
)

# Build feature order check table
step4u_34_feature_order_check_df = pd.DataFrame(
    {
        "feature_position": list(range(step4u_34_contract_feature_count)),
        "contract_feature_name": step4u_34_contract_feature_list,
        "input_matrix_feature_name": step4u_34_model_input_feature_list,
    }
)

if step4u_23_has_feature_names_in == 1:
    step4u_34_feature_order_check_df["model_object_feature_name"] = step4u_34_model_object_feature_list
    step4u_34_feature_order_check_df["input_vs_model_match"] = (
        step4u_34_feature_order_check_df["input_matrix_feature_name"]
        == step4u_34_feature_order_check_df["model_object_feature_name"]
    ).astype("int8")

display(step4u_34_model_input_summary_df)
display(step4u_34_feature_order_check_df.head(20))

,selected_scoring_date,record_count,feature_count_from_contract,feature_count_from_input_matrix,duplicate_contract_feature_count,duplicate_input_column_count,missing_batch_feature_count,contract_order_consistent,model_object_feature_count,model_object_order_consistent,model_input_ready
0,2020-10-31,326,34,34,0,0,0,1,34,1,1


,feature_position,contract_feature_name,input_matrix_feature_name,model_object_feature_name,input_vs_model_match
0,0,alert_hours_roll_sum_14,alert_hours_roll_sum_14,alert_hours_roll_sum_14,1
1,1,alert_hours_lag_1,alert_hours_lag_1,alert_hours_lag_1,1
2,2,alert_severity_roll_max_7,alert_severity_roll_max_7,alert_severity_roll_max_7,1
3,3,consecutive_alert_days_before_t,consecutive_alert_days_before_t,consecutive_alert_days_before_t,1
4,4,alert_lag_1,alert_lag_1,alert_lag_1,1
5,5,alert_lag_2,alert_lag_2,alert_lag_2,1
6,6,alert_lag_3,alert_lag_3,alert_lag_3,1
7,7,alert_lag_7,alert_lag_7,alert_lag_7,1
8,8,deficit_alert_lag_1,deficit_alert_lag_1,deficit_alert_lag_1,1
9,9,high_severity_alert_last_7d,high_severity_alert_last_7d,high_severity_alert_last_7d,1


In [13]:
# 4.3.5 Walidacja typów danych i nullowości cech modelowych

import pandas as pd
from pandas.api.types import (
    is_bool_dtype,
    is_integer_dtype,
    is_float_dtype,
    is_numeric_dtype,
)

# Validate required objects from previous steps
required_objects = [
    "step4u_22_feature_list",
    "step4u_32_selected_scoring_date",
    "step4u_34_model_input_df",
    "step4u_33_null_semantics_feature_df",
    "step4u_33_null_semantics_feature_key",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define null semantics helper columns
step4u_35_null_flag_column_candidates = [
    "null_semantics_review_flag",
    "review_flag",
    "null_review_flag",
]

step4u_35_null_label_column_candidates = [
    "null_semantics_label",
    "null_semantics_category",
    "missing_label",
    "missing_category",
    "semantics_label",
]

step4u_35_null_flag_column = next(
    (
        column_name
        for column_name in step4u_35_null_flag_column_candidates
        if column_name in step4u_33_null_semantics_feature_df.columns
    ),
    None,
)

step4u_35_null_label_column = next(
    (
        column_name
        for column_name in step4u_35_null_label_column_candidates
        if column_name in step4u_33_null_semantics_feature_df.columns
    ),
    None,
)

# Build null semantics lookup
step4u_35_null_semantics_lookup_df = step4u_33_null_semantics_feature_df[
    [step4u_33_null_semantics_feature_key]
    + ([step4u_35_null_flag_column] if step4u_35_null_flag_column is not None else [])
    + ([step4u_35_null_label_column] if step4u_35_null_label_column is not None else [])
].copy()

step4u_35_null_semantics_lookup_df = step4u_35_null_semantics_lookup_df.rename(
    columns={step4u_33_null_semantics_feature_key: "feature_name"}
)

if step4u_35_null_flag_column is not None:
    step4u_35_null_semantics_lookup_df[step4u_35_null_flag_column] = pd.to_numeric(
        step4u_35_null_semantics_lookup_df[step4u_35_null_flag_column],
        errors="coerce",
    ).fillna(0).astype("int8")

if step4u_35_null_label_column is not None:
    step4u_35_null_semantics_lookup_df[step4u_35_null_label_column] = (
        step4u_35_null_semantics_lookup_df[step4u_35_null_label_column]
        .astype("string")
    )

# Profile feature dtypes and nulls
step4u_35_feature_profile_rows = []

for feature_name in step4u_22_feature_list:
    feature_series = step4u_34_model_input_df[feature_name]
    feature_dtype = str(feature_series.dtype)

    step4u_35_feature_profile_rows.append(
        {
            "feature_name": feature_name,
            "feature_dtype": feature_dtype,
            "is_bool_dtype": int(is_bool_dtype(feature_series)),
            "is_integer_dtype": int(is_integer_dtype(feature_series)),
            "is_float_dtype": int(is_float_dtype(feature_series)),
            "is_numeric_dtype": int(is_numeric_dtype(feature_series)),
            "record_count": int(feature_series.shape[0]),
            "null_count": int(feature_series.isna().sum()),
            "null_share": float(feature_series.isna().mean()),
        }
    )

step4u_35_feature_profile_df = pd.DataFrame(step4u_35_feature_profile_rows)

# Validate model-compatible dtypes
step4u_35_feature_profile_df["model_dtype_compatible"] = (
    (
        step4u_35_feature_profile_df["is_bool_dtype"]
        + step4u_35_feature_profile_df["is_integer_dtype"]
        + step4u_35_feature_profile_df["is_float_dtype"]
    ) > 0
).astype("int8")

step4u_35_incompatible_dtype_features = step4u_35_feature_profile_df.loc[
    step4u_35_feature_profile_df["model_dtype_compatible"] == 0,
    "feature_name",
].tolist()

assert not step4u_35_incompatible_dtype_features, (
    f"Incompatible feature dtypes detected: {step4u_35_incompatible_dtype_features}"
)

# Attach null semantics metadata
step4u_35_feature_profile_df = step4u_35_feature_profile_df.merge(
    step4u_35_null_semantics_lookup_df,
    on="feature_name",
    how="left",
)

step4u_35_semantics_columns = [
    column_name
    for column_name in [step4u_35_null_flag_column, step4u_35_null_label_column]
    if column_name is not None
]

if step4u_35_semantics_columns:
    step4u_35_missing_semantics_after_merge = step4u_35_feature_profile_df.loc[
        step4u_35_feature_profile_df[step4u_35_semantics_columns].isna().all(axis=1),
        "feature_name",
    ].tolist()
else:
    step4u_35_missing_semantics_after_merge = []

assert not step4u_35_missing_semantics_after_merge, (
    f"Missing null semantics after merge: {step4u_35_missing_semantics_after_merge}"
)

# Define review-level null attention
if step4u_35_null_flag_column is not None:
    step4u_35_feature_profile_df["review_flagged_null"] = (
        (step4u_35_feature_profile_df["null_count"] > 0)
        & (step4u_35_feature_profile_df[step4u_35_null_flag_column] == 1)
    ).astype("int8")
else:
    step4u_35_feature_profile_df["review_flagged_null"] = 0

step4u_35_review_flagged_null_features = step4u_35_feature_profile_df.loc[
    step4u_35_feature_profile_df["review_flagged_null"] == 1,
    "feature_name",
].tolist()

# Build batch type and null summary
step4u_35_batch_type_null_summary_df = pd.DataFrame(
    [
        {
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "record_count": int(step4u_34_model_input_df.shape[0]),
            "feature_count": int(step4u_34_model_input_df.shape[1]),
            "model_dtype_compatible_feature_count": int(
                (step4u_35_feature_profile_df["model_dtype_compatible"] == 1).sum()
            ),
            "incompatible_dtype_feature_count": int(len(step4u_35_incompatible_dtype_features)),
            "features_with_nulls_count": int((step4u_35_feature_profile_df["null_count"] > 0).sum()),
            "review_flagged_null_feature_count": int(len(step4u_35_review_flagged_null_features)),
            "review_attention_required": int(len(step4u_35_review_flagged_null_features) > 0),
            "type_and_null_validation_ready": 1,
        }
    ]
)

# Build review-level null detail
step4u_35_review_flagged_null_df = (
    step4u_35_feature_profile_df.loc[
        step4u_35_feature_profile_df["review_flagged_null"] == 1,
        [
            column_name
            for column_name in [
                "feature_name",
                "feature_dtype",
                "null_count",
                "null_share",
                step4u_35_null_flag_column,
                step4u_35_null_label_column,
                "review_flagged_null",
            ]
            if column_name is not None
        ],
    ]
    .sort_values(["null_count", "feature_name"], ascending=[False, True])
    .reset_index(drop=True)
)

display(step4u_35_batch_type_null_summary_df)
display(step4u_35_review_flagged_null_df)

,selected_scoring_date,record_count,feature_count,model_dtype_compatible_feature_count,incompatible_dtype_feature_count,features_with_nulls_count,review_flagged_null_feature_count,review_attention_required,type_and_null_validation_ready
0,2020-10-31,326,34,34,0,1,1,1,1


,feature_name,feature_dtype,null_count,null_share,null_semantics_review_flag,null_semantics_label,review_flagged_null
0,is_cold_start,float32,13,0.039877,1,review_unexpected_missing_pattern,1


In [14]:
# 4.3.6 Walidacja anti-leakage i finalnej gotowości wsadu scoringowego

import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_31_scoring_date_mode",
    "step4u_32_selected_scoring_date",
    "step4u_32_available_dates_df",
    "step4u_33_scoring_batch_source_df",
    "step4u_34_model_input_df",
    "step4u_22_feature_list",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define forbidden non-model columns and leakage-sensitive columns
step4u_36_forbidden_model_columns = [
    "activity_date",
    "station_id",
    "split_name",
    "target_alert_day",
    "alert_hours_count",
    "deficit_hours",
    "overflow_hours",
    "alert_severity_max",
    "alert_severity_sum",
    "alert_share_of_day",
    "alert_direction_any",
]

step4u_36_forbidden_pattern_keywords = [
    "target",
    "label",
    "prediction",
    "predicted",
]

# Validate batch date scope
step4u_36_batch_unique_dates = (
    step4u_33_scoring_batch_source_df["activity_date"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert len(step4u_36_batch_unique_dates) == 1, (
    "Scoring batch must contain exactly one activity_date."
)
assert step4u_36_batch_unique_dates[0] == step4u_32_selected_scoring_date, (
    "Scoring batch date is not consistent with selected scoring date."
)

# Validate selected scoring date according to mode
if step4u_31_scoring_date_mode == "latest_available":
    assert step4u_32_selected_scoring_date == step4u_32_available_dates_df["activity_date"].max(), (
        "Latest available scoring mode is not aligned with the latest available date."
    )
else:
    assert step4u_32_selected_scoring_date in step4u_32_available_dates_df["activity_date"].tolist(), (
        "Selected scoring date is not present in available scoring dates."
    )

# Validate model input contains only contracted features
step4u_36_model_input_columns = [str(column_name) for column_name in step4u_34_model_input_df.columns]
step4u_36_contract_feature_list = [str(feature_name) for feature_name in step4u_22_feature_list]

step4u_36_unexpected_model_columns = [
    column_name
    for column_name in step4u_36_model_input_columns
    if column_name not in step4u_36_contract_feature_list
]
assert not step4u_36_unexpected_model_columns, (
    f"Unexpected columns detected in model input: {step4u_36_unexpected_model_columns}"
)

step4u_36_missing_contract_columns = [
    column_name
    for column_name in step4u_36_contract_feature_list
    if column_name not in step4u_36_model_input_columns
]
assert not step4u_36_missing_contract_columns, (
    f"Missing contracted columns in model input: {step4u_36_missing_contract_columns}"
)

# Validate leakage-sensitive columns are excluded from model input
step4u_36_forbidden_columns_present_in_batch = [
    column_name
    for column_name in step4u_36_forbidden_model_columns
    if column_name in step4u_33_scoring_batch_source_df.columns
]

step4u_36_forbidden_columns_present_in_model_input = [
    column_name
    for column_name in step4u_36_forbidden_model_columns
    if column_name in step4u_36_model_input_columns
]
assert not step4u_36_forbidden_columns_present_in_model_input, (
    f"Forbidden columns detected in model input: {step4u_36_forbidden_columns_present_in_model_input}"
)

step4u_36_keyword_flagged_model_columns = sorted(
    {
        column_name
        for column_name in step4u_36_model_input_columns
        for keyword in step4u_36_forbidden_pattern_keywords
        if keyword in column_name.lower()
    }
)
assert not step4u_36_keyword_flagged_model_columns, (
    f"Leakage-sensitive keyword columns detected in model input: {step4u_36_keyword_flagged_model_columns}"
)

# Build excluded non-model columns inventory
step4u_36_non_model_columns_in_batch = [
    column_name
    for column_name in step4u_33_scoring_batch_source_df.columns
    if column_name not in step4u_36_model_input_columns
]

step4u_36_excluded_columns_df = pd.DataFrame(
    {
        "excluded_column_name": step4u_36_non_model_columns_in_batch,
        "is_forbidden_contract_column": [
            int(column_name in step4u_36_forbidden_model_columns)
            for column_name in step4u_36_non_model_columns_in_batch
        ],
    }
).sort_values("excluded_column_name").reset_index(drop=True)

# Build anti-leakage summary
step4u_36_anti_leakage_summary_df = pd.DataFrame(
    [
        {
            "scoring_date_mode": step4u_31_scoring_date_mode,
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "record_count": int(step4u_34_model_input_df.shape[0]),
            "model_feature_count": int(len(step4u_36_model_input_columns)),
            "non_model_column_count_in_batch": int(len(step4u_36_non_model_columns_in_batch)),
            "forbidden_columns_present_in_batch_count": int(len(step4u_36_forbidden_columns_present_in_batch)),
            "forbidden_columns_present_in_model_input_count": int(len(step4u_36_forbidden_columns_present_in_model_input)),
            "keyword_flagged_model_column_count": int(len(step4u_36_keyword_flagged_model_columns)),
            "anti_leakage_ready": 1,
            "scoring_input_ready": 1,
        }
    ]
)

display(step4u_36_anti_leakage_summary_df)
display(step4u_36_excluded_columns_df)

,scoring_date_mode,selected_scoring_date,record_count,model_feature_count,non_model_column_count_in_batch,forbidden_columns_present_in_batch_count,forbidden_columns_present_in_model_input_count,keyword_flagged_model_column_count,anti_leakage_ready,scoring_input_ready
0,latest_available,2020-10-31,326,34,3,3,0,0,1,1


,excluded_column_name,is_forbidden_contract_column
0,activity_date,1
1,split_name,1
2,station_id,1


In [15]:
# 4.3.7 Zapis finalnego wsadu scoringowego i kontraktu sekcji 4.3

import json
from pathlib import Path
from datetime import date, datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_31_scoring_input_artifact_path",
    "step4u_31_scoring_input_contract_artifact_path",
    "step4u_31_model_ready_path",
    "step4u_31_null_semantics_path",
    "step4u_31_scoring_date_mode",
    "step4u_31_source_artifact_df",
    "step4u_31_scoring_date_mode_df",
    "step4u_22_decision_threshold",
    "step4u_22_feature_list",
    "step4u_32_selected_scoring_date",
    "step4u_32_scoring_date_summary_df",
    "step4u_33_scoring_batch_source_df",
    "step4u_33_scoring_batch_summary_df",
    "step4u_33_null_semantics_summary_df",
    "step4u_34_model_input_df",
    "step4u_34_model_input_summary_df",
    "step4u_35_batch_type_null_summary_df",
    "step4u_35_review_flagged_null_df",
    "step4u_36_anti_leakage_summary_df",
    "step4u_36_excluded_columns_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define JSON-safe converter
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(sub_value) for key, sub_value in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))

    if isinstance(value, pd.Series):
        return make_json_safe(value.tolist())

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if value is pd.NA:
        return None

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, float) and pd.isna(value):
        return None

    if pd.isna(value):
        return None

    return value

step4u_37_scoring_input_artifact_path = Path(step4u_31_scoring_input_artifact_path)
step4u_37_scoring_input_contract_artifact_path = Path(step4u_31_scoring_input_contract_artifact_path)

step4u_37_scoring_input_artifact_path.parent.mkdir(parents=True, exist_ok=True)
step4u_37_scoring_input_contract_artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Define final scoring input columns
step4u_37_key_columns = ["activity_date", "station_id"]
step4u_37_feature_columns = [str(feature_name) for feature_name in step4u_22_feature_list]

# Build final scoring input dataset
step4u_37_key_df = (
    step4u_33_scoring_batch_source_df[step4u_37_key_columns]
    .copy()
    .reset_index(drop=True)
)

step4u_37_feature_df = (
    step4u_34_model_input_df[step4u_37_feature_columns]
    .copy()
    .reset_index(drop=True)
)

assert step4u_37_key_df.shape[0] == step4u_37_feature_df.shape[0], (
    "Key rows and feature rows are not aligned."
)

step4u_37_scoring_input_df = pd.concat(
    [step4u_37_key_df, step4u_37_feature_df],
    axis=1,
)

step4u_37_expected_columns = step4u_37_key_columns + step4u_37_feature_columns
step4u_37_final_columns = list(step4u_37_scoring_input_df.columns)

assert step4u_37_final_columns == step4u_37_expected_columns, (
    "Final scoring input column order is not consistent with the contract."
)

step4u_37_duplicate_key_count = int(
    step4u_37_scoring_input_df.duplicated(subset=step4u_37_key_columns).sum()
)
assert step4u_37_duplicate_key_count == 0, (
    "Final scoring input contains duplicate key rows."
)

assert step4u_37_scoring_input_df["activity_date"].notna().all(), (
    "Final scoring input contains invalid activity_date values."
)
assert step4u_37_scoring_input_df["station_id"].notna().all(), (
    "Final scoring input contains missing station_id values."
)

assert int(step4u_37_scoring_input_df.shape[1]) == int(len(step4u_37_expected_columns)), (
    "Final scoring input column count is not consistent with the contract."
)

# Save scoring input parquet
step4u_37_scoring_input_table = pa.Table.from_pandas(
    step4u_37_scoring_input_df,
    preserve_index=False,
)
pq.write_table(
    step4u_37_scoring_input_table,
    step4u_37_scoring_input_artifact_path,
)

# Build scoring input contract payload
step4u_37_review_flagged_null_features = (
    step4u_35_review_flagged_null_df["feature_name"].astype("string").tolist()
    if not step4u_35_review_flagged_null_df.empty
    else []
)

step4u_37_scoring_input_contract_payload = {
    "section_name": "4.3",
    "scoring_date_mode": step4u_31_scoring_date_mode,
    "selected_scoring_date": pd.Timestamp(step4u_32_selected_scoring_date).date(),
    "source_model_ready_path": str(step4u_31_model_ready_path),
    "source_null_semantics_path": str(step4u_31_null_semantics_path),
    "decision_threshold": float(step4u_22_decision_threshold),
    "key_columns": step4u_37_key_columns,
    "feature_columns": step4u_37_feature_columns,
    "feature_count": int(len(step4u_37_feature_columns)),
    "record_count": int(step4u_37_scoring_input_df.shape[0]),
    "station_count": int(step4u_37_scoring_input_df["station_id"].nunique()),
    "excluded_non_model_columns": step4u_36_excluded_columns_df,
    "review_flagged_null_features": step4u_37_review_flagged_null_features,
    "source_artifact_inventory": step4u_31_source_artifact_df,
    "scoring_date_mode_summary": step4u_31_scoring_date_mode_df,
    "scoring_date_summary": step4u_32_scoring_date_summary_df,
    "scoring_batch_summary": step4u_33_scoring_batch_summary_df,
    "null_semantics_summary": step4u_33_null_semantics_summary_df,
    "model_input_summary": step4u_34_model_input_summary_df,
    "type_and_null_summary": step4u_35_batch_type_null_summary_df,
    "anti_leakage_summary": step4u_36_anti_leakage_summary_df,
}

step4u_37_scoring_input_contract_payload = make_json_safe(
    step4u_37_scoring_input_contract_payload
)

with open(step4u_37_scoring_input_contract_artifact_path, "w", encoding="utf-8") as file:
    json.dump(step4u_37_scoring_input_contract_payload, file, ensure_ascii=False, indent=2)

# Build artifact save summary
step4u_37_artifact_save_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4u_03_scoring_input.parquet",
            "artifact_path": str(step4u_37_scoring_input_artifact_path),
            "exists": int(step4u_37_scoring_input_artifact_path.exists()),
            "row_count": int(step4u_37_scoring_input_df.shape[0]),
            "column_count": int(step4u_37_scoring_input_df.shape[1]),
        },
        {
            "artifact_name": "b4u_03_scoring_input_contract.json",
            "artifact_path": str(step4u_37_scoring_input_contract_artifact_path),
            "exists": int(step4u_37_scoring_input_contract_artifact_path.exists()),
            "row_count": pd.NA,
            "column_count": pd.NA,
        },
    ]
)

display(step4u_37_artifact_save_df)

,artifact_name,artifact_path,exists,row_count,column_count
0,b4u_03_scoring_input.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1,326,36
1,b4u_03_scoring_input_contract.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1,<NA>,<NA>


## Podsumowanie działu 4.3 Przygotowanie wsadu danych do predykcji dziennej

**Kontekst inżynieryjny:** W dziale `4.3` przygotowano finalny wsad inferencyjny zgodny z kontraktem modelu dziennego `dzień–stacja`. Najpierw zdefiniowano źródła wsadu, czyli `b3_13_model_ready_dataset.parquet` jako zbiór model-ready oraz `b3_13_null_semantics.parquet` jako kontrakt semantyki braków danych. Następnie ustalono tryb wyboru daty scoringowej i w bieżącym uruchomieniu zastosowano tryb `latest_available`, który wskazał datę `2020-10-31` jako najnowszy dostępny batch scoringowy. W kolejnym kroku wycięto batch dla tej daty, zachowując ziarnko `dzień–stacja`, pełen zestaw cech modelowych oraz wymagane kolumny kluczowe. Następnie zweryfikowano zgodność kolejności cech pomiędzy kontraktem inferencyjnym, macierzą wejściową oraz samym obiektem modelowym. Dalej przeprowadzono walidację typów danych i nullowości, a także potwierdzono zgodność wsadu z zasadą anti-leakage poprzez wyłączenie z wejścia modelowego kolumn niemodelowych i leakage-sensitive. Na końcu zapisano artefakty sekcji `4.3`: `b4u_03_scoring_input.parquet` oraz `b4u_03_scoring_input_contract.json`.

**Interpretacja wyniku:** Źródłowy zbiór `b3_13_model_ready_dataset.parquet` zawiera `218222` rekordy i `38` kolumn, a kontrakt `b3_13_null_semantics.parquet` obejmuje `34` definicje cech modelowych. Kontrakt inferencyjny wymaga `34` cech i wszystkie `34` są obecne w źródle. Zakres dostępnych dat scoringowych obejmuje `826` batchy od `2017-05-02` do `2020-10-31`, a w bieżącym runie wybrano najnowszą dostępną datę `2020-10-31`. Batch scoringowy dla tej daty zawiera `326` rekordów i `326` unikalnych stacji, bez duplikatów klucza `activity_date + station_id`. Zbudowana macierz wejściowa ma dokładnie `34` cechy, bez brakujących kolumn i bez duplikatów, a jej kolejność jest zgodna zarówno z kontraktem inferencyjnym, jak i z samym obiektem modelowym `LGBMClassifier`. Walidacja typów danych wykazała pełną kompatybilność wszystkich `34` cech z modelem. W zakresie nullowości wykryto `1` cechę z brakami danych, czyli `is_cold_start`, dla której zarejestrowano `13` braków, co stanowi około `3.99%` batcha; jednocześnie kontrakt `null semantics` oznacza ten wzorzec jako `review_unexpected_missing_pattern`, więc został on jawnie odnotowany jako sygnał review, bez naruszenia gotowości batcha do dalszego scoringu. Walidacja anti-leakage potwierdziła, że kolumny `activity_date`, `station_id` i `split_name` pozostały poza macierzą wejściową modelu i nie przeniknęły do finalnego inputu. Finalny artefakt `b4u_03_scoring_input.parquet` został zapisany z `326` rekordami i `36` kolumnami, czyli `2` kluczami (`activity_date`, `station_id`) oraz `34` cechami modelowymi.

**Znaczenie biznesowe:** Dział `4.3` przygotowuje rzeczywisty wsad inferencyjny, na którym model dzienny może zostać uruchomiony bez dodatkowych modyfikacji danych wejściowych. Ustalenie batcha scoringowego na podstawie najnowszej dostępnej daty zapewnia spójność z warstwą model-ready i pozwala wykonywać scoring na gotowym, zmaterializowanym zbiorze, bez sztucznego przesuwania dat. Jednocześnie zachowano możliwość dalszej parametryzacji procesu dla wskazanej historycznej daty scoringowej z dostępnego zakresu. Zastosowane walidacje chronią pipeline przed silent failures wynikającymi z brakujących kolumn, zmienionej kolejności cech, niezgodnych typów danych oraz ryzyk leakage’owych. Dzięki temu wsad zapisany w `b4u_03_scoring_input.parquet` stanowi stabilną i audytowalną podstawę pod właściwy scoring w dziale `4.4`.

**Wniosek:** Dział `4.3` został domknięty poprawnie. BLOK 4 dysponuje finalnym wsadem scoringowym zgodnym z kontraktem modelu, zachowującym ziarnko `dzień–stacja`, poprawną kolejność cech, spójność z obiektem modelowym, kompletne pokrycie `null semantics` oraz zgodność z anti-leakage. Zapisane artefakty `b4u_03_scoring_input.parquet` i `b4u_03_scoring_input_contract.json` przygotowują bezpośrednie wejście do działu `4.4`, czyli uruchomienia dziennej predykcji.

## 4.4 Predykcja dzienna

In [16]:
# 4.4.1 Wczytanie wsadu scoringowego i kontraktu predykcji

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_1_outputs_dir",
    "step4u_2_model_file_path",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_decision_threshold",
    "step4u_22_feature_list",
    "step4u_23_model_object",
    "step4u_31_scoring_input_artifact_path",
    "step4u_32_selected_scoring_date",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define Section 4.4 artifacts
step4u_41_predictions_artifact_path = Path(step4u_1_outputs_dir) / "b4u_04_predictions_daily_batch.parquet"
step4u_41_prediction_summary_artifact_path = Path(step4u_1_outputs_dir) / "b4u_04_prediction_summary.json"

# Define scoring input path
step4u_41_scoring_input_path = Path(step4u_31_scoring_input_artifact_path)
assert step4u_41_scoring_input_path.exists(), (
    f"Scoring input artifact does not exist: {step4u_41_scoring_input_path}"
)

# Load scoring input dataset
step4u_41_scoring_input_table = pq.read_table(step4u_41_scoring_input_path)
step4u_41_scoring_input_df = step4u_41_scoring_input_table.to_pandas()

# Define expected columns
step4u_41_key_columns = ["activity_date", "station_id"]
step4u_41_feature_columns = [str(feature_name) for feature_name in step4u_22_feature_list]
step4u_41_expected_columns = step4u_41_key_columns + step4u_41_feature_columns

step4u_41_actual_columns = [str(column_name) for column_name in step4u_41_scoring_input_df.columns]

assert step4u_41_actual_columns == step4u_41_expected_columns, (
    "Scoring input columns are not consistent with the prediction contract."
)

# Normalize key columns
step4u_41_scoring_input_df["activity_date"] = pd.to_datetime(
    step4u_41_scoring_input_df["activity_date"],
    errors="coerce",
).dt.normalize()

step4u_41_scoring_input_df["station_id"] = (
    step4u_41_scoring_input_df["station_id"]
    .astype("string")
)

assert step4u_41_scoring_input_df["activity_date"].notna().all(), (
    "Scoring input contains invalid activity_date values."
)
assert step4u_41_scoring_input_df["station_id"].notna().all(), (
    "Scoring input contains missing station_id values."
)

# Validate selected scoring date scope
step4u_41_unique_scoring_dates = (
    step4u_41_scoring_input_df["activity_date"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert len(step4u_41_unique_scoring_dates) == 1, (
    "Scoring input must contain exactly one scoring date."
)
assert step4u_41_unique_scoring_dates[0] == step4u_32_selected_scoring_date, (
    "Scoring input date is not consistent with the selected scoring date."
)

# Validate key uniqueness
step4u_41_duplicate_key_count = int(
    step4u_41_scoring_input_df.duplicated(subset=step4u_41_key_columns).sum()
)
assert step4u_41_duplicate_key_count == 0, (
    "Scoring input contains duplicate key rows."
)

# Build prediction input matrix
step4u_41_prediction_key_df = (
    step4u_41_scoring_input_df[step4u_41_key_columns]
    .copy()
    .reset_index(drop=True)
)

step4u_41_prediction_matrix_df = (
    step4u_41_scoring_input_df[step4u_41_feature_columns]
    .copy()
    .reset_index(drop=True)
)

assert step4u_41_prediction_matrix_df.shape[1] == len(step4u_41_feature_columns), (
    "Prediction matrix feature count is not consistent with the contract."
)

# Build scoring contract summary
step4u_41_prediction_input_summary_df = pd.DataFrame(
    [
        {
            "model_name": step4u_22_model_name,
            "model_release_tag": step4u_22_model_release_tag,
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "record_count": int(step4u_41_scoring_input_df.shape[0]),
            "station_count": int(step4u_41_scoring_input_df["station_id"].nunique()),
            "feature_count": int(step4u_41_prediction_matrix_df.shape[1]),
            "decision_threshold": float(step4u_22_decision_threshold),
            "duplicate_key_count": step4u_41_duplicate_key_count,
            "prediction_input_ready": 1,
        }
    ]
)

display(step4u_41_prediction_input_summary_df)
display(step4u_41_prediction_key_df.head(10))

,model_name,model_release_tag,selected_scoring_date,record_count,station_count,feature_count,decision_threshold,duplicate_key_count,prediction_input_ready
0,lightgbm_tuned,b4_15,2020-10-31,326,326,34,0.55,0,1


,activity_date,station_id
0,2020-10-31,1
1,2020-10-31,10
2,2020-10-31,100
3,2020-10-31,101
4,2020-10-31,103
5,2020-10-31,104
6,2020-10-31,105
7,2020-10-31,106
8,2020-10-31,107
9,2020-10-31,108


In [17]:
# 4.4.2 Uruchomienie scoringu dziennego i budowa tabeli predykcji

from datetime import datetime, timezone

import numpy as np
import pandas as pd

# Validate required objects from previous steps
required_objects = [
    "step4u_22_decision_threshold",
    "step4u_23_model_object",
    "step4u_23_has_predict_proba",
    "step4u_23_has_classes",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_32_selected_scoring_date",
    "step4u_41_prediction_key_df",
    "step4u_41_prediction_matrix_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

assert step4u_23_has_predict_proba == 1, "Loaded model does not expose predict_proba."
assert step4u_23_has_classes == 1, "Loaded model does not expose classes_."

# Resolve positive class position
step4u_42_model_classes = list(step4u_23_model_object.classes_)
assert len(step4u_42_model_classes) == 2, "Prediction pipeline expects a binary classifier."

if 1 in step4u_42_model_classes:
    step4u_42_positive_class_value = 1
    step4u_42_positive_class_index = step4u_42_model_classes.index(1)
elif "1" in step4u_42_model_classes:
    step4u_42_positive_class_value = "1"
    step4u_42_positive_class_index = step4u_42_model_classes.index("1")
else:
    raise AssertionError(
        f"Positive class label 1 was not found in model classes: {step4u_42_model_classes}"
    )

# Run raw probability scoring
step4u_42_predict_proba_array = step4u_23_model_object.predict_proba(
    step4u_41_prediction_matrix_df
)

assert step4u_42_predict_proba_array.ndim == 2, (
    "predict_proba output must be a 2D array."
)
assert step4u_42_predict_proba_array.shape[0] == step4u_41_prediction_matrix_df.shape[0], (
    "Probability row count is not consistent with prediction input."
)
assert step4u_42_predict_proba_array.shape[1] == len(step4u_42_model_classes), (
    "Probability column count is not consistent with model classes."
)

step4u_42_predicted_probability = step4u_42_predict_proba_array[:, step4u_42_positive_class_index]
step4u_42_predicted_probability = np.asarray(step4u_42_predicted_probability, dtype="float64")

assert np.isfinite(step4u_42_predicted_probability).all(), (
    "Predicted probabilities contain non-finite values."
)
assert ((step4u_42_predicted_probability >= 0.0) & (step4u_42_predicted_probability <= 1.0)).all(), (
    "Predicted probabilities must be in the range [0, 1]."
)

# Build predicted label with fixed threshold
step4u_42_applied_threshold = float(step4u_22_decision_threshold)
step4u_42_predicted_label = (
    step4u_42_predicted_probability >= step4u_42_applied_threshold
).astype("int8")

# Define scoring timestamp
step4u_42_scoring_timestamp = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

# Build prediction output table
step4u_42_predictions_df = step4u_41_prediction_key_df.copy()
step4u_42_predictions_df["predicted_probability"] = step4u_42_predicted_probability.astype("float32")
step4u_42_predictions_df["predicted_label"] = step4u_42_predicted_label.astype("int8")
step4u_42_predictions_df["scoring_timestamp"] = step4u_42_scoring_timestamp
step4u_42_predictions_df["model_release_tag"] = step4u_22_model_release_tag
step4u_42_predictions_df["applied_threshold"] = step4u_42_applied_threshold

step4u_42_predictions_df = step4u_42_predictions_df[
    [
        "activity_date",
        "station_id",
        "predicted_probability",
        "predicted_label",
        "scoring_timestamp",
        "model_release_tag",
        "applied_threshold",
    ]
].copy()

step4u_42_duplicate_prediction_key_count = int(
    step4u_42_predictions_df.duplicated(
        subset=["activity_date", "station_id", "scoring_timestamp"]
    ).sum()
)
assert step4u_42_duplicate_prediction_key_count == 0, (
    "Prediction output contains duplicate prediction keys."
)

# Build prediction summary
step4u_42_prediction_summary_df = pd.DataFrame(
    [
        {
            "model_name": step4u_22_model_name,
            "model_release_tag": step4u_22_model_release_tag,
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "scoring_timestamp": step4u_42_scoring_timestamp,
            "record_count": int(step4u_42_predictions_df.shape[0]),
            "positive_prediction_count": int(step4u_42_predictions_df["predicted_label"].sum()),
            "positive_prediction_share": float(step4u_42_predictions_df["predicted_label"].mean()),
            "probability_min": float(step4u_42_predictions_df["predicted_probability"].min()),
            "probability_mean": float(step4u_42_predictions_df["predicted_probability"].mean()),
            "probability_max": float(step4u_42_predictions_df["predicted_probability"].max()),
            "applied_threshold": step4u_42_applied_threshold,
            "positive_class_label": str(step4u_42_positive_class_value),
            "prediction_ready_for_save": 1,
        }
    ]
)

display(step4u_42_prediction_summary_df)
display(step4u_42_predictions_df.head(10))

,model_name,model_release_tag,selected_scoring_date,scoring_timestamp,record_count,positive_prediction_count,positive_prediction_share,probability_min,probability_mean,probability_max,applied_threshold,positive_class_label,prediction_ready_for_save
0,lightgbm_tuned,b4_15,2020-10-31,2026-04-21T13:46:42+00:00,326,303,0.929448,0.082192,0.73389,0.951512,0.55,1,1


,activity_date,station_id,predicted_probability,predicted_label,scoring_timestamp,model_release_tag,applied_threshold
0,2020-10-31,1,0.784789,1,2026-04-21T13:46:42+00:00,b4_15,0.55
1,2020-10-31,10,0.779369,1,2026-04-21T13:46:42+00:00,b4_15,0.55
2,2020-10-31,100,0.774553,1,2026-04-21T13:46:42+00:00,b4_15,0.55
3,2020-10-31,101,0.923846,1,2026-04-21T13:46:42+00:00,b4_15,0.55
4,2020-10-31,103,0.847250,1,2026-04-21T13:46:42+00:00,b4_15,0.55
5,2020-10-31,104,0.813981,1,2026-04-21T13:46:42+00:00,b4_15,0.55
6,2020-10-31,105,0.789439,1,2026-04-21T13:46:42+00:00,b4_15,0.55
7,2020-10-31,106,0.784626,1,2026-04-21T13:46:42+00:00,b4_15,0.55
8,2020-10-31,107,0.874763,1,2026-04-21T13:46:42+00:00,b4_15,0.55
9,2020-10-31,108,0.729405,1,2026-04-21T13:46:42+00:00,b4_15,0.55


In [18]:
# 4.4.3 Zapis wyników predykcji i kontraktu sekcji 4.4

import json
from pathlib import Path
from datetime import date, datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_41_predictions_artifact_path",
    "step4u_41_prediction_summary_artifact_path",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_decision_threshold",
    "step4u_31_scoring_date_mode",
    "step4u_32_selected_scoring_date",
    "step4u_42_scoring_timestamp",
    "step4u_42_predictions_df",
    "step4u_42_prediction_summary_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define JSON-safe converter
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(sub_value) for key, sub_value in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))

    if isinstance(value, pd.Series):
        return make_json_safe(value.tolist())

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if value is pd.NA:
        return None

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, float) and pd.isna(value):
        return None

    if pd.isna(value):
        return None

    return value

step4u_43_predictions_artifact_path = Path(step4u_41_predictions_artifact_path)
step4u_43_prediction_summary_artifact_path = Path(step4u_41_prediction_summary_artifact_path)

step4u_43_predictions_artifact_path.parent.mkdir(parents=True, exist_ok=True)
step4u_43_prediction_summary_artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Validate final prediction table contract
step4u_43_expected_prediction_columns = [
    "activity_date",
    "station_id",
    "predicted_probability",
    "predicted_label",
    "scoring_timestamp",
    "model_release_tag",
    "applied_threshold",
]

step4u_43_actual_prediction_columns = list(step4u_42_predictions_df.columns)

assert step4u_43_actual_prediction_columns == step4u_43_expected_prediction_columns, (
    "Prediction output columns are not consistent with the contract."
)

step4u_43_duplicate_prediction_key_count = int(
    step4u_42_predictions_df.duplicated(
        subset=["activity_date", "station_id", "scoring_timestamp"]
    ).sum()
)
assert step4u_43_duplicate_prediction_key_count == 0, (
    "Prediction output contains duplicate prediction keys."
)

assert step4u_42_predictions_df["activity_date"].notna().all(), (
    "Prediction output contains invalid activity_date values."
)
assert step4u_42_predictions_df["station_id"].notna().all(), (
    "Prediction output contains missing station_id values."
)

assert step4u_42_predictions_df["predicted_probability"].between(0.0, 1.0).all(), (
    "Prediction output contains probabilities outside [0, 1]."
)

assert step4u_42_predictions_df["predicted_label"].isin([0, 1]).all(), (
    "Prediction output contains invalid predicted_label values."
)

assert (step4u_42_predictions_df["model_release_tag"] == step4u_22_model_release_tag).all(), (
    "Prediction output contains inconsistent model_release_tag values."
)

assert (step4u_42_predictions_df["applied_threshold"] == float(step4u_22_decision_threshold)).all(), (
    "Prediction output contains inconsistent applied_threshold values."
)

# Save predictions parquet
step4u_43_predictions_table = pa.Table.from_pandas(
    step4u_42_predictions_df,
    preserve_index=False,
)
pq.write_table(
    step4u_43_predictions_table,
    step4u_43_predictions_artifact_path,
)

# Build prediction summary payload
step4u_43_prediction_summary_payload = {
    "section_name": "4.4",
    "model_name": step4u_22_model_name,
    "model_release_tag": step4u_22_model_release_tag,
    "scoring_date_mode": step4u_31_scoring_date_mode,
    "selected_scoring_date": pd.Timestamp(step4u_32_selected_scoring_date).date(),
    "scoring_timestamp": step4u_42_scoring_timestamp,
    "applied_threshold": float(step4u_22_decision_threshold),
    "record_count": int(step4u_42_predictions_df.shape[0]),
    "station_count": int(step4u_42_predictions_df["station_id"].nunique()),
    "positive_prediction_count": int(step4u_42_predictions_df["predicted_label"].sum()),
    "positive_prediction_share": float(step4u_42_predictions_df["predicted_label"].mean()),
    "probability_min": float(step4u_42_predictions_df["predicted_probability"].min()),
    "probability_mean": float(step4u_42_predictions_df["predicted_probability"].mean()),
    "probability_max": float(step4u_42_predictions_df["predicted_probability"].max()),
    "prediction_summary_table": step4u_42_prediction_summary_df,
}

step4u_43_prediction_summary_payload = make_json_safe(
    step4u_43_prediction_summary_payload
)

with open(step4u_43_prediction_summary_artifact_path, "w", encoding="utf-8") as file:
    json.dump(step4u_43_prediction_summary_payload, file, ensure_ascii=False, indent=2)

# Build artifact save summary
step4u_43_artifact_save_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4u_04_predictions_daily_batch.parquet",
            "artifact_path": str(step4u_43_predictions_artifact_path),
            "exists": int(step4u_43_predictions_artifact_path.exists()),
            "row_count": int(step4u_42_predictions_df.shape[0]),
            "column_count": int(step4u_42_predictions_df.shape[1]),
        },
        {
            "artifact_name": "b4u_04_prediction_summary.json",
            "artifact_path": str(step4u_43_prediction_summary_artifact_path),
            "exists": int(step4u_43_prediction_summary_artifact_path.exists()),
            "row_count": pd.NA,
            "column_count": pd.NA,
        },
    ]
)

display(step4u_43_artifact_save_df)

,artifact_name,artifact_path,exists,row_count,column_count
0,b4u_04_predictions_daily_batch.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1,326,7
1,b4u_04_prediction_summary.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1,<NA>,<NA>


## Podsumowanie działu 4.4 Predykcja dzienna

**Kontekst inżynieryjny:** W dziale `4.4` uruchomiono scoring dzienny dla gotowego wsadu `dzień–stacja` zapisanego wcześniej w `b4u_03_scoring_input.parquet`. Najpierw wczytano wsad scoringowy i zbudowano macierz predykcyjną zgodną z kontraktem modelu, zachowując `34` cechy wejściowe oraz klucze `activity_date` i `station_id`. Następnie wykorzystano załadowany wcześniej model `lightgbm_tuned` z pakietu `b4_15`, aby policzyć `raw probability` dla klasy pozytywnej `1`. Na tej podstawie zbudowano `predicted_label` przy użyciu sztywnego progu `0.55` pobranego z kontraktu inferencyjnego. W tabeli wynikowej dodano również metadane traceability: `scoring_timestamp`, `model_release_tag` oraz `applied_threshold`. Na końcu zapisano artefakty sekcji `4.4`: `b4u_04_predictions_daily_batch.parquet` oraz `b4u_04_prediction_summary.json`.

**Interpretacja wyniku:** Scoring został policzony dla batcha z datą `2020-10-31`, obejmującego `326` rekordów i `326` unikalnych stacji. Model użyty do inferencji to `lightgbm_tuned`, powiązany z pakietem `b4_15`. Dla każdego rekordu policzono `predicted_probability`, a następnie utworzono `predicted_label` przy progu `0.55`. Łącznie `303` rekordy otrzymały etykietę dodatnią, co odpowiada udziałowi około `92.94%` batcha. Najniższe przewidywane prawdopodobieństwo wyniosło około `0.0822`, średnie około `0.7339`, a najwyższe około `0.9515`. Wynikowa tabela predykcji została zapisana jako `b4u_04_predictions_daily_batch.parquet` i zawiera `326` rekordów oraz `7` kolumn: `activity_date`, `station_id`, `predicted_probability`, `predicted_label`, `scoring_timestamp`, `model_release_tag`, `applied_threshold`. Dodatkowo zapisano opisowy kontrakt podsumowujący scoring w pliku `b4u_04_prediction_summary.json`.

**Znaczenie biznesowe:** Dział `4.4` materializuje właściwy wynik modelu dziennego i zamienia przygotowany wcześniej wsad inferencyjny na rzeczywiste predykcje gotowe do dalszego użycia. Dzięki zastosowaniu stałego progu `0.55` wynik jest spójny z kontraktem inferencyjnym i nie zależy od bieżącej, uznaniowej regulacji parametrów. Dodane metadane traceability sprawiają, że każda predykcja może być jednoznacznie powiązana z konkretnym momentem scoringu, użytym pakietem modelowym i zastosowanym progiem decyzji. To wzmacnia kontrolę wersji, audytowalność i czytelność procesu wdrożeniowego, a także przygotowuje wynik do bezpośredniego przekazania do warstwy prezentacyjnej aplikacji.

**Wniosek:** Dział `4.4` został domknięty poprawnie. BLOK 4 dysponuje już kompletnym wynikiem scoringu dziennego, zapisanym zarówno w formie tabeli predykcji, jak i w formie opisowego podsumowania. Artefakty `b4u_04_predictions_daily_batch.parquet` i `b4u_04_prediction_summary.json` stanowią gotowe wejście do działu `4.5`, czyli zapisu wyników do warstwy wyjściowej i ich prezentacji w aplikacji.

## 4.5 Zapis wyników i ich prezentacja w aplikacji

In [19]:
# 4.5.1 Przygotowanie zbioru wynikowego dla aplikacji

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_1_outputs_dir",
    "step4u_41_predictions_artifact_path",
    "step4u_42_predictions_df",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_decision_threshold",
    "step4u_32_selected_scoring_date",
    "step4u_33_scoring_batch_source_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define Section 4.5 artifacts
step4u_51_predictions_for_app_artifact_path = Path(step4u_1_outputs_dir) / "b4u_05_predictions_for_app.parquet"
step4u_51_app_handoff_artifact_path = Path(step4u_1_outputs_dir) / "b4u_05_app_handoff.json"

# Load saved prediction artifact
step4u_51_predictions_path = Path(step4u_41_predictions_artifact_path)
assert step4u_51_predictions_path.exists(), (
    f"Prediction artifact does not exist: {step4u_51_predictions_path}"
)

step4u_51_predictions_table = pq.read_table(step4u_51_predictions_path)
step4u_51_predictions_saved_df = step4u_51_predictions_table.to_pandas()

step4u_51_predictions_saved_df["activity_date"] = pd.to_datetime(
    step4u_51_predictions_saved_df["activity_date"],
    errors="coerce",
).dt.normalize()

step4u_51_predictions_saved_df["station_id"] = (
    step4u_51_predictions_saved_df["station_id"]
    .astype("string")
)

assert step4u_51_predictions_saved_df["activity_date"].notna().all(), (
    "Saved predictions contain invalid activity_date values."
)
assert step4u_51_predictions_saved_df["station_id"].notna().all(), (
    "Saved predictions contain missing station_id values."
)

# Define app context columns
step4u_51_app_context_candidates = [
    "hub_flag",
    "persona",
    "microzone",
    "is_cold_start",
    "is_holiday",
    "is_business_free_day",
    "alert_hours_roll_sum_14",
    "alert_hours_lag_1",
    "consecutive_alert_days_before_t",
]

step4u_51_app_context_columns = [
    column_name
    for column_name in step4u_51_app_context_candidates
    if column_name in step4u_33_scoring_batch_source_df.columns
]

# Build app context frame
step4u_51_app_context_df = (
    step4u_33_scoring_batch_source_df[
        ["activity_date", "station_id"] + step4u_51_app_context_columns
    ]
    .copy()
    .reset_index(drop=True)
)

step4u_51_app_context_df["activity_date"] = pd.to_datetime(
    step4u_51_app_context_df["activity_date"],
    errors="coerce",
).dt.normalize()

step4u_51_app_context_df["station_id"] = (
    step4u_51_app_context_df["station_id"]
    .astype("string")
)

# Merge predictions with app context
step4u_51_predictions_for_app_df = step4u_51_predictions_saved_df.merge(
    step4u_51_app_context_df,
    on=["activity_date", "station_id"],
    how="left",
    validate="one_to_one",
)

# Define app output order
step4u_51_app_output_columns = [
    "activity_date",
    "station_id",
    "predicted_probability",
    "predicted_label",
    "scoring_timestamp",
    "model_release_tag",
    "applied_threshold",
] + step4u_51_app_context_columns

step4u_51_predictions_for_app_df = step4u_51_predictions_for_app_df[
    step4u_51_app_output_columns
].copy()

step4u_51_predictions_for_app_df = step4u_51_predictions_for_app_df.sort_values(
    ["predicted_probability", "station_id"],
    ascending=[False, True],
).reset_index(drop=True)

# Validate final app dataset
step4u_51_duplicate_key_count = int(
    step4u_51_predictions_for_app_df.duplicated(
        subset=["activity_date", "station_id", "scoring_timestamp"]
    ).sum()
)
assert step4u_51_duplicate_key_count == 0, (
    "App dataset contains duplicate prediction keys."
)

assert step4u_51_predictions_for_app_df["activity_date"].nunique() == 1, (
    "App dataset must contain exactly one scoring date."
)
assert step4u_51_predictions_for_app_df["activity_date"].iloc[0] == step4u_32_selected_scoring_date, (
    "App dataset scoring date is not consistent with selected scoring date."
)

# Build app dataset summary
step4u_51_app_dataset_summary_df = pd.DataFrame(
    [
        {
            "model_name": step4u_22_model_name,
            "model_release_tag": step4u_22_model_release_tag,
            "selected_scoring_date": step4u_32_selected_scoring_date,
            "record_count": int(step4u_51_predictions_for_app_df.shape[0]),
            "station_count": int(step4u_51_predictions_for_app_df["station_id"].nunique()),
            "positive_prediction_count": int(step4u_51_predictions_for_app_df["predicted_label"].sum()),
            "context_column_count": int(len(step4u_51_app_context_columns)),
            "duplicate_key_count": step4u_51_duplicate_key_count,
            "app_dataset_ready": 1,
        }
    ]
)

display(step4u_51_app_dataset_summary_df)
display(step4u_51_predictions_for_app_df.head(10))

,model_name,model_release_tag,selected_scoring_date,record_count,station_count,positive_prediction_count,context_column_count,duplicate_key_count,app_dataset_ready
0,lightgbm_tuned,b4_15,2020-10-31,326,326,303,7,0,1


,activity_date,station_id,predicted_probability,predicted_label,scoring_timestamp,model_release_tag,applied_threshold,hub_flag,is_cold_start,is_holiday,is_business_free_day,alert_hours_roll_sum_14,alert_hours_lag_1,consecutive_alert_days_before_t
0,2020-10-31,201,0.951512,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,272.0,18.0,69.0
1,2020-10-31,16,0.932096,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,275.0,17.0,48.0
2,2020-10-31,5,0.927936,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,241.0,17.0,2.0
3,2020-10-31,64,0.927556,1,2026-04-21T13:46:42+00:00,b4_15,0.55,1,0.0,1,1,260.0,18.0,11.0
4,2020-10-31,113,0.925940,1,2026-04-21T13:46:42+00:00,b4_15,0.55,1,0.0,1,1,283.0,7.0,61.0
5,2020-10-31,101,0.923846,1,2026-04-21T13:46:42+00:00,b4_15,0.55,1,0.0,1,1,248.0,17.0,12.0
6,2020-10-31,547,0.923610,1,2026-04-21T13:46:42+00:00,b4_15,0.55,1,0.0,1,1,291.0,23.0,36.0
7,2020-10-31,58,0.920777,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,245.0,19.0,55.0
8,2020-10-31,63,0.908855,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,257.0,23.0,40.0
9,2020-10-31,4,0.903446,1,2026-04-21T13:46:42+00:00,b4_15,0.55,0,0.0,1,1,252.0,17.0,11.0


In [20]:
# 4.5.2 Zapis zbioru wynikowego dla aplikacji i handoff sekcji 4.5

import json
from pathlib import Path
from datetime import date, datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Validate required objects from previous steps
required_objects = [
    "step4u_1_app_path",
    "step4u_1_outputs_dir",
    "step4u_2_feature_importance_path",
    "step4u_22_model_name",
    "step4u_22_model_release_tag",
    "step4u_22_decision_threshold",
    "step4u_31_scoring_date_mode",
    "step4u_32_selected_scoring_date",
    "step4u_32_available_dates_df",
    "step4u_51_predictions_for_app_artifact_path",
    "step4u_51_app_handoff_artifact_path",
    "step4u_51_predictions_for_app_df",
    "step4u_51_app_dataset_summary_df",
]

missing_objects = [name for name in required_objects if name not in globals()]
assert not missing_objects, f"Missing required objects from previous steps: {missing_objects}"

# Define JSON-safe converter
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(key): make_json_safe(sub_value) for key, sub_value in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))

    if isinstance(value, pd.Series):
        return make_json_safe(value.tolist())

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if value is pd.NA:
        return None

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, float) and pd.isna(value):
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    return value

step4u_52_predictions_for_app_artifact_path = Path(step4u_51_predictions_for_app_artifact_path)
step4u_52_app_handoff_artifact_path = Path(step4u_51_app_handoff_artifact_path)

step4u_52_predictions_for_app_artifact_path.parent.mkdir(parents=True, exist_ok=True)
step4u_52_app_handoff_artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Define app dataset contract
step4u_52_base_prediction_columns = [
    "activity_date",
    "station_id",
    "predicted_probability",
    "predicted_label",
    "scoring_timestamp",
    "model_release_tag",
    "applied_threshold",
]

step4u_52_expected_optional_columns = [
    "hub_flag",
    "persona",
    "microzone",
    "is_cold_start",
    "is_holiday",
    "is_business_free_day",
    "alert_hours_roll_sum_14",
    "alert_hours_lag_1",
    "consecutive_alert_days_before_t",
]

step4u_52_context_columns = [
    column_name
    for column_name in step4u_52_expected_optional_columns
    if column_name in step4u_51_predictions_for_app_df.columns
]

step4u_52_expected_columns = step4u_52_base_prediction_columns + step4u_52_context_columns
step4u_52_actual_columns = list(step4u_51_predictions_for_app_df.columns)

assert step4u_52_actual_columns == step4u_52_expected_columns, (
    "App dataset columns are not consistent with the expected app contract."
)

# Validate app dataset
assert step4u_51_predictions_for_app_df["activity_date"].notna().all(), (
    "App dataset contains invalid activity_date values."
)
assert step4u_51_predictions_for_app_df["station_id"].notna().all(), (
    "App dataset contains missing station_id values."
)
assert step4u_51_predictions_for_app_df["predicted_probability"].between(0.0, 1.0).all(), (
    "App dataset contains probabilities outside [0, 1]."
)
assert step4u_51_predictions_for_app_df["predicted_label"].isin([0, 1]).all(), (
    "App dataset contains invalid predicted_label values."
)

step4u_52_duplicate_key_count = int(
    step4u_51_predictions_for_app_df.duplicated(
        subset=["activity_date", "station_id", "scoring_timestamp"]
    ).sum()
)
assert step4u_52_duplicate_key_count == 0, (
    "App dataset contains duplicate prediction keys."
)

assert step4u_51_predictions_for_app_df["activity_date"].nunique() == 1, (
    "App dataset must contain exactly one scoring date."
)
assert pd.Timestamp(step4u_51_predictions_for_app_df["activity_date"].iloc[0]).normalize() == pd.Timestamp(step4u_32_selected_scoring_date).normalize(), (
    "App dataset scoring date is not consistent with selected scoring date."
)

assert (step4u_51_predictions_for_app_df["model_release_tag"] == step4u_22_model_release_tag).all(), (
    "App dataset contains inconsistent model_release_tag values."
)
assert (step4u_51_predictions_for_app_df["applied_threshold"] == float(step4u_22_decision_threshold)).all(), (
    "App dataset contains inconsistent applied_threshold values."
)

# Save app dataset parquet
step4u_52_predictions_for_app_table = pa.Table.from_pandas(
    step4u_51_predictions_for_app_df,
    preserve_index=False,
)
pq.write_table(
    step4u_52_predictions_for_app_table,
    step4u_52_predictions_for_app_artifact_path,
)

# Build app handoff payload
step4u_52_available_scoring_date_min = pd.Timestamp(
    step4u_32_available_dates_df["activity_date"].min()
).date()
step4u_52_available_scoring_date_max = pd.Timestamp(
    step4u_32_available_dates_df["activity_date"].max()
).date()

step4u_52_explainability_columns = [
    column_name
    for column_name in [
        "alert_hours_roll_sum_14",
        "alert_hours_lag_1",
        "consecutive_alert_days_before_t",
    ]
    if column_name in step4u_51_predictions_for_app_df.columns
]

step4u_52_app_handoff_payload = {
    "section_name": "4.5",
    "app_file_name": Path(step4u_1_app_path).name,
    "app_file_path": str(step4u_1_app_path),
    "app_technology": "streamlit",
    "model_name": step4u_22_model_name,
    "model_release_tag": step4u_22_model_release_tag,
    "selected_scoring_date": pd.Timestamp(step4u_32_selected_scoring_date).date(),
    "scoring_date_mode": step4u_31_scoring_date_mode,
    "supported_scoring_modes": ["latest_available", "selected_date"],
    "available_scoring_date_min": step4u_52_available_scoring_date_min,
    "available_scoring_date_max": step4u_52_available_scoring_date_max,
    "available_scoring_date_count": int(step4u_32_available_dates_df.shape[0]),
    "applied_threshold": float(step4u_22_decision_threshold),
    "record_count": int(step4u_51_predictions_for_app_df.shape[0]),
    "station_count": int(step4u_51_predictions_for_app_df["station_id"].nunique()),
    "positive_prediction_count": int(step4u_51_predictions_for_app_df["predicted_label"].sum()),
    "positive_prediction_share": float(step4u_51_predictions_for_app_df["predicted_label"].mean()),
    "predictions_for_app_path": str(step4u_52_predictions_for_app_artifact_path),
    "feature_importance_path": str(step4u_2_feature_importance_path),
    "app_output_columns": step4u_52_actual_columns,
    "context_columns": step4u_52_context_columns,
    "explainability_columns": step4u_52_explainability_columns,
    "sort_order": [
        {"column": "predicted_probability", "ascending": False},
        {"column": "station_id", "ascending": True},
    ],
    "filters_supported": [
        "station_id",
        "hub_flag",
        "predicted_label",
        "is_cold_start",
        "is_holiday",
    ],
    "technical_panel_source": {
        "feature_importance_file": "b4_15_feature_importance.parquet",
        "ranking_column": "consensus_rank",
    },
    "app_dataset_summary": step4u_51_app_dataset_summary_df,
}

step4u_52_app_handoff_payload = make_json_safe(step4u_52_app_handoff_payload)

with open(step4u_52_app_handoff_artifact_path, "w", encoding="utf-8") as file:
    json.dump(step4u_52_app_handoff_payload, file, ensure_ascii=False, indent=2)

# Build artifact save summary
step4u_52_artifact_save_df = pd.DataFrame(
    [
        {
            "artifact_name": "b4u_05_predictions_for_app.parquet",
            "artifact_path": str(step4u_52_predictions_for_app_artifact_path),
            "exists": int(step4u_52_predictions_for_app_artifact_path.exists()),
            "row_count": int(step4u_51_predictions_for_app_df.shape[0]),
            "column_count": int(step4u_51_predictions_for_app_df.shape[1]),
        },
        {
            "artifact_name": "b4u_05_app_handoff.json",
            "artifact_path": str(step4u_52_app_handoff_artifact_path),
            "exists": int(step4u_52_app_handoff_artifact_path.exists()),
            "row_count": pd.NA,
            "column_count": pd.NA,
        },
    ]
)

display(step4u_52_artifact_save_df)

,artifact_name,artifact_path,exists,row_count,column_count
0,b4u_05_predictions_for_app.parquet,/root/projects/6_samodzielny_projekt_koncowy_w...,1,326,14
1,b4u_05_app_handoff.json,/root/projects/6_samodzielny_projekt_koncowy_w...,1,<NA>,<NA>


## Podsumowanie działu 4.5 Zapis wyników i ich prezentacja w aplikacji

**Kontekst inżynieryjny:** W dziale `4.5` przygotowano warstwę wyjściową przeznaczoną bezpośrednio dla aplikacji dziennej. Najpierw wczytano zapisane wcześniej wyniki scoringu z `b4u_04_predictions_daily_batch.parquet`, a następnie połączono je z kontekstem operacyjnym pochodzącym z batcha scoringowego, obejmującym wybrane flagi i cechy pomocnicze do prezentacji wyników użytkownikowi. W rezultacie zbudowano dataset app-ready zawierający nie tylko właściwe predykcje modelu, ale również dodatkowe kolumny potrzebne do filtrowania, sortowania i lekkiego wyjaśniania wyniku w aplikacji. Następnie zapisano ten zbiór w `b4u_05_predictions_for_app.parquet` oraz utworzono handoff aplikacyjny w `b4u_05_app_handoff.json`, opisujący kontrakt danych, wspierane tryby pracy aplikacji, źródło feature importance oraz podstawowe parametry działania interfejsu.

**Interpretacja wyniku:** Zbiór wynikowy dla aplikacji został zapisany poprawnie i zawiera `326` rekordów oraz `14` kolumn. Każdy rekord opisuje jedną stację dla daty scoringu `2020-10-31` i zawiera komplet informacji potrzebnych do prezentacji wyniku: `activity_date`, `station_id`, `predicted_probability`, `predicted_label`, `scoring_timestamp`, `model_release_tag`, `applied_threshold` oraz `7` kolumn kontekstowych. Do kolumn kontekstowych należą: `hub_flag`, `is_cold_start`, `is_holiday`, `is_business_free_day`, `alert_hours_roll_sum_14`, `alert_hours_lag_1` oraz `consecutive_alert_days_before_t`. Zbiór nie zawiera duplikatów klucza predykcyjnego, zachowuje jedną spójną datę scoringową oraz jest posortowany malejąco po `predicted_probability`, dzięki czemu najwyższe ryzyka są od razu widoczne na początku tabeli. W pliku `b4u_05_app_handoff.json` zapisano dodatkowo metadane potrzebne do aplikacji, w tym: nazwę pliku aplikacji `aplikacja_dzien_stacja.py`, technologię `streamlit`, aktywny tryb scoringu, wspierane tryby `latest_available` i `selected_date`, zakres dostępnych dat scoringowych, kolumny kontekstowe, kolumny explainability oraz źródło rankingu ważności cech `b4_15_feature_importance.parquet` z kolumną `consensus_rank`.

**Znaczenie biznesowe:** Dział `4.5` tworzy warstwę prezentacyjną, która zamienia wynik modelu w dane gotowe do użycia przez użytkownika końcowego. Dzięki dołączeniu flag takich jak `hub_flag`, `is_cold_start` czy `is_holiday` aplikacja nie pokazuje wyłącznie samego prawdopodobieństwa, ale umożliwia szybszą interpretację i priorytetyzację stacji w warunkach operacyjnych. Dodanie prostych cech kontekstowych, takich jak `alert_hours_roll_sum_14`, `alert_hours_lag_1` i `consecutive_alert_days_before_t`, wzmacnia zaufanie do wyniku bez dokładania ciężkiej interpretowalności online. Zapis `b4u_05_app_handoff.json` ułatwia późniejsze zbudowanie aplikacji Streamlit, ponieważ formalizuje oczekiwany układ danych, obsługiwane tryby pracy i źródła metadanych technicznych. To sprawia, że warstwa użytkowa BLOKU 4 jest nie tylko czytelna, ale też przygotowana do wdrożenia w lekkiej aplikacji MVP.

**Wniosek:** Dział `4.5` został domknięty poprawnie. BLOK 4 dysponuje już gotowym zbiorem wynikowym dla aplikacji `b4u_05_predictions_for_app.parquet` oraz handoffem aplikacyjnym `b4u_05_app_handoff.json`. Oznacza to, że pipeline wdrożeniowy został zamknięty od wsadu inferencyjnego aż do warstwy prezentacyjnej, a kolejnym naturalnym krokiem jest implementacja pliku `aplikacja_dzien_stacja.py` w technologii Streamlit na podstawie przygotowanych artefaktów i kontraktów.